# Post-hoc mechanistic tests

Downstream analyses following:
- `grpDotChangeSurfParcel.ipynb` and `grpDotContrastsSurfParcel.ipynb` — group-averaged HRFs and vs-baseline t-tests
- `indDotTfceTTestSurfParcelSpatiotemporal4fcpt.ipynb` — individual-level TFCE tests, masking, and group-level direction-consistency/directional-hit-count inference

These analyses ask *why* the group-level direction-consistency/directional-hit-count tests found strong 6mo habituation but no comparable 12mo effect, given that individual-level habituation rates are similar at both ages.

## Section 1 — target habituation parcels
| Analysis | What it tests |
|---|---|
| **1.** Group-averaged HRFs at 6mo vs 12mo in the target habituation parcels | Is the group-averaged effect preserved despite direction-consistency/directional-hit-count failure at 12mo? |
| **1 i.** Latency shift: Fam1 vs Contrast Epoch peak times per infant, by age | Does Contrast Epoch peak later than Fam1 at 12mo but not 6mo? |
| **1 ii.** Early-vs-late window comparison of Fam1 and Contrast Epoch | At 12mo: Fam1 > Contrast Epoch in early window, Fam1 < Contrast Epoch in late window? |
| **1 iii.** Within-parcel Fam1–Contrast Epoch waveform correlation (wide window) | Do Fam1 and Contrast Epoch have the same shape at 6mo (high correlation) but different shapes at 12mo (lower correlation)? |
| **1 iv.** Diagnostic: are per-infant per-parcel traces usable? | Sanity check on 1 i-iii null results |
| **1 v.** Tight-window Fam1-Contrast Epoch correlation | Recover shape similarity by removing pre/post-response noise |
| **1 vi.** Lag-optimised Fam1-Contrast Epoch correlation | Distinguish shape mismatch from timing shift |

## Section 2 — within-vs-between infant variability at target parcels
| Analysis | What it tests |
|---|---|
| **2.** Within-infant vs between-infant response variability | Attention (within) vs individuation (between) at target parcels |

## Section 3 — personal-ROI follow-up
Sections 1 and 2 hit a signal-to-noise ceiling at the per-parcel per-infant level. Section 3 rebuilds the same analyses at an infant-specific ROI (the set of parcels where that infant showed a significant response at each age), which pools signal across parcels and should substantially improve SNR at the individual level. It also enables new spatial-individuation tests.

| Analysis | What it tests |
|---|---|
| **3.1** Build personal ROIs at both thresholds | ROI definition |
| **3.2** Compute per-infant ROI HRFs (Fam1 and Contrast Epoch) | Per-infant ROI-averaged waveforms |
| **3.3** ROI-level data quality diagnostic (analog of 1 iv) | Whether ROI averaging fixed the SNR problem |
| **3.4** ROI-level group-averaged HRFs, 6mo vs 12mo (analog of 1) | Load-bearing plot at the ROI level |
| **3.5** ROI size distribution per age | How much cortex does each infant activate? |
| **3.6** Habituation amplitude per infant | Does the infant show habituation in their own ROI? |
| **3.7** ROI-level Fam1-Contrast Epoch waveform correlation (wide, tight, lag-opt; analogs of 1 iii, 1 v, 1 vi) | Shape similarity of Fam1 vs Contrast Epoch with better SNR |
| **3.8** ROI-level latency (analog of 1 i) | Peak time shift with better SNR |
| **3.9** ROI-level within-vs-between infant variability (analog of 2) | Mechanism-adjudicating test with better SNR |
| **3.10** ROI overlap between infants — Jaccard similarity | Do infants share the same ROI parcels? |
| **3.11** ROI centre-of-mass scatter | Is the scatter of ROI centres larger at 12mo? |
| **3.12** Parcel-level infant-count distribution | Are 6mo activations concentrated at "hotspot" parcels while 12mo are diffuse — explaining the group-test failure? |
| **3.13** ROI-level direction-consistency/directional-hit-count analog | Does the population habituate when each infant contributes one ROI-aggregated observation? |
| **3.14** Where does the habituation occur (spatial map) | Descriptive parcel-hotspot map (A) and ROI-restricted inferential per-parcel test (B) |
| **3.15** Group HRFs at BH-significant parcels | Are the parcel-level habituation effects (from 3.14b) driven by canonical HRF shapes, or by noise? |
| **3.16** Spatial map of BH-significant parcels | Do the parcels cluster in expected auditory cortex or are they scattered? Which parcels are shared across ages? |

## Section 4 — mixed-effects group-level habituation

Section 3 shows that individual-level habituation is present and equivalent at both ages. Section 4 formalises this as a proper group-level test that accounts for the sources of variance the direction-consistency/directional-hit-count parcel-level test could not: infant clustering, parcel-to-parcel variance, and per-observation SNR.

| Analysis | What it tests |
|---|---|
| **4.1** Assemble mixed-effects dataset | One row per (Infant, Age, Parcel) with hab_amp and SNR |
| **4.2** Fit mixed-effects model | `hab_amp ~ Age + SNR + (1|Infant) + (1|Parcel)` |
| **4.3** Report fixed effects and interpretation | Intercept (population habituation), Age (developmental difference), SNR (measurement quality control) |

## Imports and paths

In [ ]:
import h5py
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import nibabel as nib
    _NIB_OK = True
except ImportError as e:
    print(f"Skipping 3.11: {e}")
    _NIB_OK = False
import plotly.graph_objects as go
import statsmodels.formula.api as smf
import statsmodels.api as sm
import sys
from collections import defaultdict, Counter
from mpl_toolkits.mplot3d import Axes3D  
from pathlib import Path
from scipy.io import loadmat
from scipy.spatial import cKDTree
from scipy.stats import binomtest, wilcoxon, mannwhitneyu
from statsmodels.stats.multitest import multipletests

# --- Paths ---
DATA = "/Users/sambe/Documents/Python_Projects/FDA/data/"

HRF_CSV   = DATA + "group_averaged_HRFs_by_epoch.csv"
GROUP_CSV = DATA + "GROUPLEVEL_direction_tests_all_tests.csv"
ACT_CSV   = DATA + "surf_parcel_TFCE_circshift_results_INDIV_spatiotemporal_FDR.csv"

# OPTIONS FOR SCRIPT RUNNING
# --- Chromophore switch ---
# For chromophore specific testing/visualisation - see corresponding document for HbR
CHROM = 1
# Switch the threshold here and re-run section 3 to swap between FDR and uncorrected.
ROI_THRESHOLD = 'fdr'       # 'fdr' or 'uncorrected'


# --- Sampling / windowing (matches grpDotChangeSurfParcel.ipynb) ---
FS_HZ       = 10               # sampling rate
STIM_ONSET  = 40               # sample index of stimulus onset
TARGET_LEN  = 231              # total samples per trace
PEAK_ONSET  = 110               # sample index of peak-window start (9s post-stim)
PEAK_OFFSET = 170              # sample index of peak-window end   (13s post-stim)

# Time axis in seconds relative to stim onset
TIME_AXIS = (np.arange(TARGET_LEN) - STIM_ONSET) / FS_HZ

CHROM_LABELS = {1: 'HbO', 2: 'HbR'}
CHROM_COLOURS = {1: 'crimson', 2: 'royalblue'}

# --- Chromophore switch ---
# For chromophore specific testing/visualisation - see corresponding document for HbR
CHROM = 2

# Switch the threshold here and re-run section 3 to swap between FDR and uncorrected.
ROI_THRESHOLD = 'fdr'       # 'fdr' or 'uncorrected'

# --- Habituation contrast switch ---
# 'fam1_fam3'  = Fam1 (trials 1-5)  vs Fam3   (trials 11-15) — standard habituation
# 'delayed'    = Fam1 (trials 1-5)  vs Novel  (trials 16-20) — delayed habituation
# 'long_range' = Fam1 (trials 1-5)  vs Post   (trials 21-25) — long-range habituation
HAB_CONTRAST = 'long_range'  # 'fam1_fam2', 'fam1_fam3', 'delayed', 'long_range'

CONTRAST_TRIALS = {
    'fam1_fam2':  {'group_a': list(range(1, 6)),  'group_b': list(range(6, 11)),  'label_b': 'Fam2'},
    'fam1_fam3':  {'group_a': list(range(1, 6)),  'group_b': list(range(11, 16)), 'label_b': 'Fam3'},
    'delayed':    {'group_a': list(range(1, 6)),  'group_b': list(range(16, 21)), 'label_b': 'Novel'},
    'long_range': {'group_a': list(range(1, 6)),  'group_b': list(range(21, 26)), 'label_b': 'Post'},
}
GROUP_A_TRIALS = CONTRAST_TRIALS[HAB_CONTRAST]['group_a']
GROUP_B_TRIALS = CONTRAST_TRIALS[HAB_CONTRAST]['group_b']
LABEL_B        = CONTRAST_TRIALS[HAB_CONTRAST]['label_b']

# Map HAB_CONTRAST to the 'test' name in GROUPLEVEL_direction_tests_all_tests.csv
CONTRAST_TO_TEST_NAME = {
    'fam1_fam2': 'earlyHab',   
    'fam1_fam3': 'habituation',
    'delayed':   'delayedHab',
    'long_range': 'lrHab',
}
CONTRAST_TEST_NAME = CONTRAST_TO_TEST_NAME[HAB_CONTRAST]

Save outputs as .txt file

In [ ]:
REPORTS = Path("/Users/sambe/Documents/Python_Projects/FDA/reports/")
OUTPUT_TXT = REPORTS / ("postHocMechanisticReport_HbR_" 
                        + CONTRAST_TO_TEST_NAME[HAB_CONTRAST] 
                        + ".txt")

# Start a fresh report each run
OUTPUT_TXT.write_text("POST-HOC MECHANISTIC REPORT\n\n")

def report(*args, **kwargs):
    text = kwargs.get("sep", " ").join(str(a) for a in args)
    end = kwargs.get("end", "\n")

    print(*args, **kwargs)

    with open(OUTPUT_TXT, "a") as f:
        f.write(text + end)

report(f"Chromophore: " + CHROM_LABELS[CHROM] + "; Condition Contrast: " + CONTRAST_TO_TEST_NAME[HAB_CONTRAST] + "; Threshold: " + ROI_THRESHOLD)
report("=========================================")
report()

## 1. Group-averaged HRFs in target parcels — 6mo vs 12mo

**Target set**: parcels where 6mo HbO habituation reached the directional-hit-count test FDR significance (*NOTE: weaker criterion for HbR than for HbO*)

**Supplementary set**: parcels where 12mo activation reached both direction-consistency and directional-hit-count. This provides a reference for what a healthy 12mo group HRF looks like in genuinely-activated cortex.

**Key result:** at the target parcels, 6mo shows canonical habituation (Fam1 > Contrast Epoch in peak window) but 12mo shows *timing/shape differences* between Fam1 and Contrast Epoch rather than magnitude habituation. This motivates the follow-up tests below.


In [ ]:
# --- Load ---
hrfDf   = pd.read_csv(HRF_CSV)
groupDf = pd.read_csv(GROUP_CSV)

report(f"Loaded HRF table: {len(hrfDf):,} rows")
report(f"Loaded group-level table: {len(groupDf):,} rows")
report(f"\nHRF unique (Age, Parcel, Chrom, Epoch) combos: "
      f"{hrfDf.groupby(['Age','Parcel','Chrom','Epoch']).ngroups:,}")
report(f"HRF epochs available: {sorted(hrfDf['Epoch'].unique())}")

In [ ]:
# --- Define target and supplementary parcel sets ---

# Target: 6mo HbO habituation, both group-level tests FDR-significant
target_mask = (
    (groupDf['test']  == CONTRAST_TEST_NAME) &
    (groupDf['Age']   == 6) &
    (groupDf['Chrom'] == CHROM) &   
    (groupDf['sig_direction_consistency_fdr']) &
    (groupDf['sig_directional_hit_count_fdr'])
)
target_parcels = sorted(groupDf.loc[target_mask, 'Parcel'].unique().tolist())

# Supplementary: 12mo activation, both direction-consistency and directional-hit-count (union of HbO and HbR)
supp_mask = (
    (groupDf['test'] == 'activation') &
    (groupDf['Age']  == 12) &
    (groupDf['sig_direction_consistency_fdr']) &
    (groupDf['sig_directional_hit_count_fdr'])
)
supp_df = groupDf.loc[supp_mask, ['Parcel', 'Chrom']].drop_duplicates()
supp_parcels_hbO = sorted(supp_df.loc[supp_df['Chrom']==1, 'Parcel'].unique().tolist())
supp_parcels_hbR = sorted(supp_df.loc[supp_df['Chrom']==2, 'Parcel'].unique().tolist())

report(f"Target set (6mo {CHROM_LABELS[CHROM]} {HAB_CONTRAST}, directional-hit-count): {len(target_parcels)} parcels")
report(f"  Parcels: {target_parcels}")
report(f"\nSupplementary set (12mo activation, direction-consistency and directional-hit-count):")
report(f"  HbO: {len(supp_parcels_hbO)} parcels")
report(f"  HbR: {len(supp_parcels_hbR)} parcels")

In [ ]:
# --- Colour and style scheme ---
# HbO: shades of red. HbR: shades of blue.
# EARLY GROUP: solid, darker shade.
# LATE GROUP:  dashed, lighter shade.
EPOCH_STYLES = {
    ('HbO', 'Fam1'):  {'color': '#8B0000', 'linestyle': '-'},
    ('HbO', 'Fam3'):  {'color': '#FF6961', 'linestyle': '--'},
    ('HbO', 'Novel'): {'color': '#FF6961', 'linestyle': '--'},
    ('HbO', 'Post'):  {'color': '#FF6961', 'linestyle': '--'},
    ('HbR', 'Fam1'):  {'color': '#00008B', 'linestyle': '-'},
    ('HbR', 'Fam3'):  {'color': '#6495ED', 'linestyle': '--'},
    ('HbR', 'Novel'): {'color': '#6495ED', 'linestyle': '--'},
    ('HbR', 'Post'):  {'color': '#6495ED', 'linestyle': '--'},
}


def _style(chrom, epoch):
    label = CHROM_LABELS.get(chrom, 'HbR')
    return EPOCH_STYLES.get((label, epoch), {'color': 'black', 'linestyle': '-'})


def get_hrf(hrf_df, age, parcel, chrom, epoch):
    """Extract mean/sem/n_infants for one (age, parcel, chrom, epoch); return None if missing."""
    sub = hrf_df[
        (hrf_df['Age']    == age) &
        (hrf_df['Parcel'] == parcel) &
        (hrf_df['Chrom']  == chrom) &
        (hrf_df['Epoch']  == epoch)
    ].sort_values('sample_idx')
    if sub.empty:
        return None
    return {
        'mean': sub['mean'].values,
        'sem':  sub['sem'].values,
        'n':    int(sub['n_infants'].iloc[0]),
    }


def plot_hrf_pair(ax, hrf_df, age, parcel, chrom, epochs=('Fam1', 'Fam3')):
    """Plot mean +/- SEM for two epochs at one (age, parcel, chrom)."""
    ns = {}
    for epoch in epochs:
        h = get_hrf(hrf_df, age, parcel, chrom, epoch)
        if h is None:
            continue
        s = _style(chrom, epoch)
        ax.plot(TIME_AXIS, h['mean'], label=epoch, lw=1.5, **s)
        ax.fill_between(TIME_AXIS, h['mean']-h['sem'], h['mean']+h['sem'],
                        color=s['color'], alpha=0.2, lw=0)
        ns[epoch] = h['n']
    ax.axhline(0, color='grey', lw=0.5, alpha=0.7)
    ax.axvline(0, color='grey', lw=0.5, ls=':', alpha=0.7)
    ax.axvspan((PEAK_ONSET - STIM_ONSET)/FS_HZ,
               (PEAK_OFFSET - STIM_ONSET)/FS_HZ,
               color='yellow', alpha=0.08, lw=0)
    n_str = ' | '.join(f"{e}: n={n}" for e, n in ns.items())
    ax.set_title(f"Parcel {parcel}  ({n_str})", fontsize=9)
    ax.set_xlabel('Time (s)')

In [ ]:
# --- Per-parcel plots: 6mo vs 12mo for the target parcels ---
# One row per parcel, two columns (6mo | 12mo).
# All panels share the same y-axis so amplitudes are directly comparable.

if target_parcels:
    n_parcels = len(target_parcels)
    fig, axes = plt.subplots(n_parcels, 2, figsize=(9, 2.2*n_parcels), sharey=True)
    if n_parcels == 1:
        axes = axes[np.newaxis, :]

    # find global y-limits so both ages share scale
    y_vals = []
    for parcel in target_parcels:
        for age in (6, 12):
            for epoch in ('Fam1', LABEL_B):
                h = get_hrf(hrfDf, age, parcel, CHROM, epoch)
                if h is not None:
                    y_vals.append(h['mean'] + h['sem'])
                    y_vals.append(h['mean'] - h['sem'])
    y_all = np.concatenate(y_vals) if y_vals else np.array([0.0])
    y_lim = (np.nanmin(y_all)*1.1, np.nanmax(y_all)*1.1)

    for i, parcel in enumerate(target_parcels):
        plot_hrf_pair(axes[i, 0], hrfDf, age=6,  parcel=parcel, chrom=CHROM)
        plot_hrf_pair(axes[i, 1], hrfDf, age=12, parcel=parcel, chrom=CHROM)
        axes[i, 0].set_ylabel(f'Δ {CHROM_LABELS[CHROM]} (μM)')
        axes[i, 0].set_ylim(y_lim)
        if i == 0:
            axes[i, 0].set_title(f"6mo — {axes[i, 0].get_title()}", fontsize=9)
            axes[i, 1].set_title(f"12mo — {axes[i, 1].get_title()}", fontsize=9)
            axes[i, 0].legend(loc='upper right', fontsize=8, frameon=False)

    fig.suptitle(f'Target {HAB_CONTRAST} parcels (6mo {CHROM_LABELS[CHROM]}, the directional-hit-count test): '
                 'group-averaged HRFs, 6mo vs 12mo', fontsize=11, y=1.0)
    fig.tight_layout()
    plt.show()
else:
    report("No target parcels found — nothing to plot.")

In [ ]:
# --- Compact summary: mean across parcels, 6mo vs 12mo ---
# Mean across parcels of the group-averaged HRF, with SEM band computed
# across parcels (between-parcel variability of the group mean).

def average_hrfs_over_parcels(hrf_df, age, parcels, chrom, epoch):
    """Return per-sample mean and SEM across parcels of the group-averaged HRF."""
    curves = []
    for p in parcels:
        h = get_hrf(hrf_df, age, p, chrom, epoch)
        if h is not None:
            curves.append(h['mean'])
    if not curves:
        return None, None
    stack = np.vstack(curves)
    m = np.nanmean(stack, axis=0)
    s = np.nanstd(stack, axis=0) / np.sqrt(stack.shape[0])
    return m, s


if target_parcels:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
    for ax, age in zip(axes, (6, 12)):
        for epoch in ('Fam1', LABEL_B):
            mean_trace, sem_trace = average_hrfs_over_parcels(
                hrfDf, age, target_parcels, CHROM, epoch
            )
            if mean_trace is None:
                continue
            s = _style(CHROM, epoch)
            ax.plot(TIME_AXIS, mean_trace, label=epoch, lw=1.8, **s)
            ax.fill_between(TIME_AXIS,
                            mean_trace - sem_trace,
                            mean_trace + sem_trace,
                            color=s['color'], alpha=0.2, lw=0)
        ax.axhline(0, color='grey', lw=0.5, alpha=0.7)
        ax.axvline(0, color='grey', lw=0.5, ls=':', alpha=0.7)
        ax.axvspan((PEAK_ONSET - STIM_ONSET)/FS_HZ,
                   (PEAK_OFFSET - STIM_ONSET)/FS_HZ,
                   color='yellow', alpha=0.10, lw=0)
        ax.set_title(f'{age}mo — averaged across {len(target_parcels)} target parcels',
                     fontsize=10)
        ax.set_xlabel('Time (s)')
        ax.legend(loc='upper right', fontsize=9, frameon=False)
    axes[0].set_ylabel(f'Δ {CHROM_LABELS[CHROM]} (μM)')
    fig.suptitle(f'Target {HAB_CONTRAST} parcels: {CHROM_LABELS[CHROM]}, mean of group-averaged HRFs '
                 '(shaded band = SEM across parcels)', y=1.02)
    fig.tight_layout()
    plt.show()

In [ ]:
# --- Supplementary set: 12mo activation-significant parcels ---
# Reference for what a healthy 12mo group HRF looks like in genuinely-activated
# cortex, averaged across parcels. SEM band = between-parcel variability.

def _plot_supp(ax, chrom, parcel_list, age=12):
    for epoch in ('Fam1', LABEL_B):
        mean_trace, sem_trace = average_hrfs_over_parcels(
            hrfDf, age, parcel_list, chrom, epoch
        )
        if mean_trace is None:
            continue
        s = _style(chrom, epoch)
        ax.plot(TIME_AXIS, mean_trace, label=epoch, lw=1.8, **s)
        ax.fill_between(TIME_AXIS,
                        mean_trace - sem_trace,
                        mean_trace + sem_trace,
                        color=s['color'], alpha=0.2, lw=0)
    ax.axhline(0, color='grey', lw=0.5, alpha=0.7)
    ax.axvline(0, color='grey', lw=0.5, ls=':', alpha=0.7)
    ax.axvspan((PEAK_ONSET - STIM_ONSET)/FS_HZ,
               (PEAK_OFFSET - STIM_ONSET)/FS_HZ,
               color='yellow', alpha=0.10, lw=0)
    ax.set_xlabel('Time (s)')
    ax.legend(loc='upper right', fontsize=9, frameon=False)


if supp_parcels_hbO:
    fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
    _plot_supp(ax, chrom=1, parcel_list=supp_parcels_hbO)
    ax.set_title(f'12mo HbO — averaged across {len(supp_parcels_hbO)} '
                 f'12mo activation-significant parcels', fontsize=10)
    ax.set_ylabel('Δ HbO (μM)')
    fig.tight_layout()
    plt.show()

if supp_parcels_hbR:
    fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))
    _plot_supp(ax, chrom=2, parcel_list=supp_parcels_hbR)
    ax.set_title(f'12mo HbR — averaged across {len(supp_parcels_hbR)} '
                 f'12mo activation-significant parcels', fontsize=10)
    ax.set_ylabel('Δ HbR (μM)')
    fig.tight_layout()
    plt.show()

### 1 i. Latency shift: Fam1 vs Contrast Epoch peak times per infant, by age

Compute, for each (Infant, Age, Parcel) at the target parcels: the sample of maximum HRF amplitude within an extended post-stim window (samples 40-170 = 0-13s post-stim), separately for Fam1 and Contrast Epoch, for HbR. Plot the *within-parcel* difference (Contrast Epoch peak time - Fam1 peak time) per infant, by age.

**Prediction under the timing-shift interpretation:**
- 6mo: HbO peak-time differences centred on 0 (Fam1 and Contrast Epoch peak at the same time).
- 12mo: HbO peak-time differences skewed positive (Contrast Epoch peaks later than Fam1).

This test uses the raw per-infant per-trial time series - needs to be recomputed here since the group-HRF CSV averages across infants. Loads the `.h5` trial dataset and metadata.


In [ ]:
# --- Load trial-level data and metadata for the per-infant analyses ---

PARCEL_META_CSV = "/Users/sambe/Documents/Python_Projects/FDA/dataRaw/indigoSurfParcelMetadataStandard.csv"
PARCEL_TS_H5    = "/Users/sambe/Documents/Python_Projects/FDA/dataRaw/indigoSurfParcelTrialsVlenStandard.h5"

metadataDf = pd.read_csv(PARCEL_META_CSV)

# Infants whose HaND task was (partly) repeated across runs (overlapping trial numbers).
# KEEP_FIRST_RUN_ONLY = True  -> keep the first run only, so every trial is a first presentation.
# KEEP_FIRST_RUN_ONLY = False -> pool runs.
# Must match grpDotContrastsSurfParcel.ipynb and indDotTfceTTestSurfParcelSpatiotemporal4fcpt.ipynb.
# Split sessions (no overlapping trials) are always kept whole.
KEEP_FIRST_RUN_ONLY = True

runTrials = metadataDf.groupby(['Age', 'Infant', 'Run'])['Trial'].apply(set)
repeated = []
for (age, infant), g in runTrials.groupby(level=[0, 1]):
    if len(g) > 1 and set.intersection(*g.values):
        repeated.append((age, infant, g.index.get_level_values('Run').min()))
print(f"Repeated-run infants: {[(a, i) for a, i, _ in repeated]}")
if KEEP_FIRST_RUN_ONLY:
    for age, infant, firstRun in repeated:
        drop = (metadataDf['Age'] == age) & (metadataDf['Infant'] == infant) & (metadataDf['Run'] != firstRun)
        metadataDf = metadataDf[~drop]
    print('  -> first run kept for these infants')
else:
    print('  -> runs pooled for these infants')

with h5py.File(PARCEL_TS_H5, 'r') as h5f:
    tsDataset = h5f['timeSeries'][()].copy()

report(f"metadataDf: {len(metadataDf):,} rows")
report(f"tsDataset:  {len(tsDataset):,} trial time series")


In [ ]:
# --- 1 i. Latency shift analysis ---
LATENCY_WINDOW = slice(40, 170)   # 0-13s post-stim

def _infant_trial_mean(inf_meta_df, epoch_trials, ts_all, target_len=231):
    curves = []
    epoch_meta = inf_meta_df[inf_meta_df['Trial'].isin(epoch_trials)]
    for idx in epoch_meta['tsIDx']:
        ts = ts_all[idx] * 1e6
        if np.isnan(ts).all():
            continue
        if len(ts) > target_len:
            ts = ts[:target_len]
        elif len(ts) < target_len:
            ts = np.pad(ts, (0, target_len - len(ts)), constant_values=np.nan)
        curves.append(ts)
    if not curves:
        return None
    return np.nanmean(curves, axis=0)


latency_rows = []
for age in (6, 12):
    age_meta = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Chrom'] == CHROM)]
    for parcel in target_parcels:
        parcel_meta = age_meta[age_meta['Parcel'] == parcel]
        for infant, inf_df in parcel_meta.groupby('Infant'):
            fam1 = _infant_trial_mean(inf_df, GROUP_A_TRIALS, tsDataset)
            fam3 = _infant_trial_mean(inf_df, GROUP_B_TRIALS, tsDataset)
            if fam1 is None or fam3 is None:
                continue
            f1w = fam1[LATENCY_WINDOW]; f3w = fam3[LATENCY_WINDOW]
            if np.all(np.isnan(f1w)) or np.all(np.isnan(f3w)):
                continue
            f1_peak_idx = int(np.nanargmin(f1w))
            f3_peak_idx = int(np.nanargmin(f3w))
            f1_peak_s = (f1_peak_idx + LATENCY_WINDOW.start - STIM_ONSET) / FS_HZ
            f3_peak_s = (f3_peak_idx + LATENCY_WINDOW.start - STIM_ONSET) / FS_HZ
            latency_rows.append({
                'Age': age, 'Parcel': parcel, 'Infant': infant,
                'Fam1_peak_s': f1_peak_s, f'{LABEL_B}_peak_s': f3_peak_s,
                'diff_s': f3_peak_s - f1_peak_s,
            })

latencyDf = pd.DataFrame(latency_rows)
if not np.size(latencyDf) == 0:
    report(f"Latency dataset: {len(latencyDf):,} infant x parcel rows")
    report(latencyDf.groupby('Age')['diff_s'].agg(['count', 'mean', 'median', 'std']))
else:
    report("Latency dataset is empty — no valid infant x parcel rows found.")


In [ ]:
# --- Plot: distribution of Contrast Epoch-Fam1 peak time differences, by age ---
if not latencyDf.empty:
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
    for ax, age in zip(axes, (6, 12)):
        sub = latencyDf[latencyDf['Age'] == age]['diff_s']
        ax.hist(sub, bins=25, color=('crimson' if age == 6 else 'firebrick'),
                alpha=0.7, edgecolor='white')
        ax.axvline(0, color='k', ls='--', lw=1, alpha=0.7)
        ax.axvline(sub.median(), color='black', lw=1.5, label=f'median={sub.median():.2f}s')
        ax.set_title(f'{age}mo  (n = {len(sub)} infant x parcel)', fontsize=10)
        ax.set_xlabel(f'{LABEL_B} peak time - Fam1 peak time (s)')
        ax.legend(fontsize=9, frameon=False)
    axes[0].set_ylabel('Count')
    fig.suptitle(f'1 i. Latency shift: {LABEL_B} peaks later than Fam1?', y=1.02)
    fig.tight_layout()
    plt.show()

    for age in (6, 12):
        diffs = latencyDf[latencyDf['Age'] == age]['diff_s'].dropna()
        if len(diffs) > 0:
            w = wilcoxon(diffs, alternative='greater')
            report(f"{age}mo: n={len(diffs)}, W={w.statistic:.1f}, p={w.pvalue:.4g}, "
                  f"median diff = {diffs.median():.2f}s")


### 1 ii. Early-vs-late window comparison

For each (Infant, Age, Parcel), average Fam1 and Contrast Epoch traces over an *early* window (5-9s post-stim) and a *late* window (10-14s post-stim). Compare the Fam1-Contrast Epoch difference in each window.

**Prediction under the timing-shift interpretation:**
- 6mo: Fam1 > Contrast Epoch in BOTH windows.
- 12mo: Fam1 > Contrast Epoch in the *early* window; Fam1 <= Contrast Epoch in the *late* window.


In [ ]:
# --- 1 ii. Early-vs-late window Fam1 vs Contrast Epoch ---
EARLY_WINDOW = slice(90, 130)    # 5-9s post-stim
LATE_WINDOW  = slice(140, 180)   # 10-14s post-stim

def _window_mean(trace, window):
    if trace is None:
        return np.nan
    w = trace[window]
    return float(np.nanmean(w)) if np.any(~np.isnan(w)) else np.nan


early_late_rows = []
for age in (6, 12):
    age_meta = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Chrom'] == CHROM)]
    for parcel in target_parcels:
        parcel_meta = age_meta[age_meta['Parcel'] == parcel]
        for infant, inf_df in parcel_meta.groupby('Infant'):
            fam1 = _infant_trial_mean(inf_df, GROUP_A_TRIALS, tsDataset)
            fam3 = _infant_trial_mean(inf_df, GROUP_B_TRIALS, tsDataset)
            if fam1 is None or fam3 is None:
                continue
            row = {
                'Age': age, 'Parcel': parcel, 'Infant': infant,
                'Fam1_early': _window_mean(fam1, EARLY_WINDOW),
                f'{LABEL_B}_early': _window_mean(fam3, EARLY_WINDOW),
                'Fam1_late':  _window_mean(fam1, LATE_WINDOW),
                f'{LABEL_B}_late':  _window_mean(fam3, LATE_WINDOW),
            }
            row['diff_early'] = row['Fam1_early'] - row[f'{LABEL_B}_early']
            row['diff_late']  = row['Fam1_late']  - row[f'{LABEL_B}_late']
            early_late_rows.append(row)

earlyLateDf = pd.DataFrame(early_late_rows)
if not np.size(earlyLateDf) == 0:
    report(f"Early/late dataset: {len(earlyLateDf):,} infant x parcel rows")
    report(earlyLateDf.groupby('Age')[['diff_early', 'diff_late']].agg(['count','mean','median']))
else:
    report("Early/late dataset is empty — no valid infant x parcel rows found.")


In [ ]:
# --- Plot: distribution of Fam1-Contrast Epoch differences in early vs late windows ---
if not earlyLateDf.empty:
    fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharey='row', sharex=True)
    for col_idx, age in enumerate((6, 12)):
        sub = earlyLateDf[earlyLateDf['Age'] == age]
        for row_idx, (col, label) in enumerate([('diff_early', 'Early window (5-9s)'),
                                                 ('diff_late',  'Late window (10-14s)')]):
            ax = axes[row_idx, col_idx]
            vals = sub[col].dropna()
            ax.hist(vals, bins=25, color=('crimson' if age == 6 else 'firebrick'),
                    alpha=0.7, edgecolor='white')
            ax.axvline(0, color='k', ls='--', lw=1, alpha=0.7)
            ax.axvline(vals.median(), color='black', lw=1.5,
                       label=f'median={vals.median():.3f}')
            ax.set_title(f'{age}mo  {label}  (n = {len(vals)})', fontsize=9)
            ax.set_xlabel(f'Fam1 - {LABEL_B} ({label.split()[0]})')
            ax.legend(fontsize=8, frameon=False)
    axes[0, 0].set_ylabel('Count'); axes[1, 0].set_ylabel('Count')
    fig.suptitle(f'1 ii. Early-vs-late window: Fam1 - {LABEL_B} by age', y=1.00)
    fig.tight_layout(); plt.show()

    for age in (6, 12):
        for col in ('diff_early', 'diff_late'):
            vals = earlyLateDf[earlyLateDf['Age'] == age][col].dropna()
            if len(vals) < 5: continue
            w = wilcoxon(vals)
            report(f"{age}mo {col}: n={len(vals)}, median={vals.median():.4f}, "
                  f"Wilcoxon vs 0 p={w.pvalue:.4g}")


### 1 iii. Fam1-Contrast Epoch waveform correlation

For each (Infant, Age, Parcel), compute the Pearson correlation between the Fam1 and contrast group waveforms across the post-stim window (samples 40-190 = 0-15s post-stim). Compare distributions across ages.

**Prediction under the timing-shift interpretation:**
- 6mo: high correlation (~0.9+) - same shape, differing only in amplitude.
- 12mo: lower correlation - shapes differ.


In [ ]:
# --- 1 iii. Fam1-Contrast Epoch waveform correlation ---
CORR_WINDOW = slice(40, 190)

def _pearson(a, b):
    a = np.asarray(a); b = np.asarray(b)
    mask = ~(np.isnan(a) | np.isnan(b))
    if mask.sum() < 3: return np.nan
    a = a[mask]; b = b[mask]
    if np.std(a) == 0 or np.std(b) == 0: return np.nan
    return float(np.corrcoef(a, b)[0, 1])


corr_rows = []
for age in (6, 12):
    age_meta = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Chrom'] == CHROM)]
    for parcel in target_parcels:
        parcel_meta = age_meta[age_meta['Parcel'] == parcel]
        for infant, inf_df in parcel_meta.groupby('Infant'):
            fam1 = _infant_trial_mean(inf_df, GROUP_A_TRIALS, tsDataset)
            fam3 = _infant_trial_mean(inf_df, GROUP_B_TRIALS, tsDataset)
            if fam1 is None or fam3 is None: continue
            r = _pearson(fam1[CORR_WINDOW], fam3[CORR_WINDOW])
            corr_rows.append({'Age': age, 'Parcel': parcel, 'Infant': infant, 'corr': r})

corrDf = pd.DataFrame(corr_rows)
if not np.size(corrDf) == 0:
    report(f"Correlation dataset: {len(corrDf):,} rows")
    report(corrDf.groupby('Age')['corr'].agg(['count','mean','median','std']))
else:
    report("Correlation dataset is empty — no valid infant x parcel rows found.")


In [ ]:
# --- Plot: distribution of Fam1-contrast correlations by age ---
if not corrDf.empty:
    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    bins = np.linspace(-1, 1, 41)
    for age, colour in [(6, 'crimson'), (12, 'firebrick')]:
        vals = corrDf[corrDf['Age'] == age]['corr'].dropna()
        ax.hist(vals, bins=bins, color=colour, alpha=0.55, edgecolor='white',
                label=f'{age}mo  (n={len(vals)}, median r={vals.median():.2f})')
    ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
    ax.set_xlabel(f'Fam1-{LABEL_B} waveform correlation (r)')
    ax.set_ylabel('Count')
    ax.set_title(f'1 iii. Fam1-{LABEL_B} waveform correlation, by age (target parcels)')
    ax.legend(fontsize=9, frameon=False)
    fig.tight_layout(); plt.show()

    c6  = corrDf[corrDf['Age'] == 6]['corr'].dropna()
    c12 = corrDf[corrDf['Age'] == 12]['corr'].dropna()
    if len(c6) > 0 and len(c12) > 0:
        u = mannwhitneyu(c6, c12, alternative='greater')
        report(f"Mann-Whitney U (6mo > 12mo): U={u.statistic:.1f}, p={u.pvalue:.4g}")
        report(f"  6mo:  n={len(c6)},  median r = {c6.median():.3f}")
        report(f"  12mo: n={len(c12)}, median r = {c12.median():.3f}")


### 1 iv. Diagnostic: are per-infant per-parcel traces usable?

The waveform correlations in 1a iii came back at median r ~= 0.07 for both ages - nowhere near 1.0 despite the group-averaged HRFs in Figure 2 clearly showing coherent habituation shapes at 6mo. This mismatch needs explaining before we take the null results at face value.

Two possibilities:
- **A. Genuinely noisy single-infant data.** 5 trials per epoch is too few to recover a clean HRF from single-parcel fNIRS. Group-average habituation is a population-level phenomenon that isn't visible in any single infant x parcel.
- **B. Analysis artefact.** Something in how per-infant means are computed is corrupting the correlations.

This cell computes diagnostics per (Infant, Age, Parcel):
- **n_valid_Fam1**, **n_valid_{LABEL_B}**: how many of the 5 trials in each epoch had usable data
- **snr_Fam1**, **snr_{LABEL_B}**: peak amplitude / baseline SD (a rough per-trace SNR)
- **peak_Fam1**, **peak_{LABEL_B}**: absolute peak amplitude in the peak window

If most rows have 5/5 valid trials AND reasonable SNR, interpretation A holds. If many rows have 2-3 valid trials or SNR near 1, interpretation B holds.


In [ ]:
# --- 1 iv. Per-row diagnostics ---
def _count_valid_and_snr(inf_meta_df, epoch_trials, ts_all,
                          baseline_slice=slice(0, 40),
                          peak_slice=slice(120, 160),
                          target_len=231):
    trials = []
    epoch_meta = inf_meta_df[inf_meta_df['Trial'].isin(epoch_trials)]
    for idx in epoch_meta['tsIDx']:
        ts = ts_all[idx] * 1e6
        if np.isnan(ts).all(): continue
        if len(ts) > target_len:
            ts = ts[:target_len]
        elif len(ts) < target_len:
            ts = np.pad(ts, (0, target_len - len(ts)), constant_values=np.nan)
        trials.append(ts)

    n_valid = len(trials)
    if n_valid == 0:
        return 0, np.nan, np.nan
    mean_trace = np.nanmean(trials, axis=0)
    baseline_seg = mean_trace[baseline_slice]
    peak_seg = mean_trace[peak_slice]
    if np.all(np.isnan(baseline_seg)) or np.all(np.isnan(peak_seg)):
        return n_valid, np.nan, np.nan
    baseline_sd = np.nanstd(baseline_seg)
    peak_amp = np.nanmax(np.abs(peak_seg - np.nanmean(baseline_seg)))
    snr = peak_amp / baseline_sd if baseline_sd > 0 else np.nan
    return n_valid, snr, peak_amp


diag_rows = []
for age in (6, 12):
    age_meta = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Chrom'] == CHROM)]
    for parcel in target_parcels:
        parcel_meta = age_meta[age_meta['Parcel'] == parcel]
        for infant, inf_df in parcel_meta.groupby('Infant'):
            n1, snr1, pk1 = _count_valid_and_snr(inf_df, GROUP_A_TRIALS, tsDataset)
            n3, snr3, pk3 = _count_valid_and_snr(inf_df, GROUP_B_TRIALS, tsDataset)
            diag_rows.append({
                'Age': age, 'Parcel': parcel, 'Infant': infant,
                'n_valid_Fam1': n1, f'n_valid_{LABEL_B}': n3,
                'snr_Fam1': snr1,   f'snr_{LABEL_B}': snr3,
                'peak_Fam1': pk1,   f'peak_{LABEL_B}': pk3,
            })

diagDf = pd.DataFrame(diag_rows)
if not np.size(diagDf) == 0:
    report(f"Diagnostic dataset: {len(diagDf):,} infant x parcel rows")
    report("\n=== Valid trials per row (of 5) ===")
    for age in (6, 12):
        sub = diagDf[diagDf['Age'] == age]
        report(f"\n{age}mo:")
        for label, col in [('Fam1', 'n_valid_Fam1'), (f'{LABEL_B}', f'n_valid_{LABEL_B}')]:
            counts = ' | '.join(f"n={k}: {(sub[col]==k).sum()}" for k in range(6))
            report(f"  {label}: {counts}")

    report("\n=== SNR (peak / baseline SD) ===")
    report(diagDf.groupby('Age')[['snr_Fam1', f'snr_{LABEL_B}']].describe().T)
else:
    report("Diagnostic dataset is empty — no valid infant x parcel rows found.")


In [ ]:
if not np.size(diagDf) == 0:        
    # --- Plot diagnostics ---
    fig, axes = plt.subplots(2, 2, figsize=(11, 6))
    for col_idx, age in enumerate((6, 12)):
        sub = diagDf[diagDf['Age'] == age]
        ax = axes[0, col_idx]
        ax.hist([sub['n_valid_Fam1'].dropna(), sub[f'n_valid_{LABEL_B}'].dropna()],
                bins=np.arange(-0.5, 6.5, 1), label=['Fam1', LABEL_B],
                color=['#00008B', '#6495ED'], alpha=0.75, edgecolor='white')
        ax.set_title(f'{age}mo: valid trials per (Infant x Parcel)', fontsize=10)
        ax.set_xlabel('Number of valid trials (of 5)')
        ax.set_ylabel('Count'); ax.legend(frameon=False, fontsize=9)
        ax.set_xticks(range(6))

    for col_idx, age in enumerate((6, 12)):
        sub = diagDf[diagDf['Age'] == age]
        ax = axes[1, col_idx]
        bins = np.linspace(0, 15, 40)
        ax.hist([sub['snr_Fam1'].dropna(), sub[f'snr_{LABEL_B}'].dropna()],
                bins=bins, label=['Fam1', LABEL_B],
                color=['#00008B', '#6495ED'], alpha=0.75, edgecolor='white')
        ax.axvline(1, color='k', ls=':', lw=1, alpha=0.7, label='SNR=1 (noise floor)')
        ax.axvline(3, color='k', ls='--', lw=1, alpha=0.7, label='SNR=3 (usable)')
        ax.set_title(f'{age}mo: SNR (peak / baseline SD)', fontsize=10)
        ax.set_xlabel('SNR'); ax.set_ylabel('Count')
        ax.legend(frameon=False, fontsize=8)

    fig.suptitle('1 iv. Per-infant per-parcel data quality diagnostics', y=1.00)
    fig.tight_layout(); plt.show()

    report("\nFraction of rows with SNR >= 3 (usable individual traces):")
    for age in (6, 12):
        sub = diagDf[diagDf['Age'] == age]
        for col in ('snr_Fam1', f'snr_{LABEL_B}'):
            vals = sub[col].dropna()
            pct = 100 * (vals >= 3).mean()
            report(f"  {age}mo {col}: {pct:.1f}% of {len(vals)} rows have SNR >= 3")


### 1 v. Tight-window Fam1-Contrast Epoch correlation

The 1a iii correlations used an aggressive 0-15s post-stim window, which includes a lot of pre-response baseline noise. If the response only lives in ~5-13s post-stim, correlating over the full window dilutes any real shape agreement with noise on either side.

This test recomputes the Fam1-contrast waveform correlation over a **tight response window (5-13s post-stim)** only. If the population-level Fam1 > Contrast Epoch pattern seen in the group-averaged HRFs (Figure 2) reflects a shared shape across infants (with amplitude habituation on top), then correlations here should be substantially higher than the ~0.07 seen with the wide window - especially at 6mo where the group HRFs clearly show canonical habituation.

**Interpretation:**
- If both ages tighten to r ~= 0.6-0.9 -> individual-level shape is real and shared; wide-window correlations were noise-diluted.
- If both ages stay near r ~= 0 -> individual data at these parcels really is dominated by noise; group habituation is an emergent statistical phenomenon.


In [ ]:
# --- 1 v. Tight-window Fam1-Contrast Epoch correlation ---
TIGHT_CORR_WINDOW = slice(90, 170)   # 5-13s post-stim

tight_corr_rows = []
for age in (6, 12):
    age_meta = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Chrom'] == CHROM)]
    for parcel in target_parcels:
        parcel_meta = age_meta[age_meta['Parcel'] == parcel]
        for infant, inf_df in parcel_meta.groupby('Infant'):
            fam1 = _infant_trial_mean(inf_df, GROUP_A_TRIALS, tsDataset)
            fam3 = _infant_trial_mean(inf_df, GROUP_B_TRIALS, tsDataset)
            if fam1 is None or fam3 is None:
                continue
            r = _pearson(fam1[TIGHT_CORR_WINDOW], fam3[TIGHT_CORR_WINDOW])
            tight_corr_rows.append({
                'Age': age, 'Parcel': parcel, 'Infant': infant, 'corr': r,
            })

tightCorrDf = pd.DataFrame(tight_corr_rows)
if not np.size(tightCorrDf) == 0:
    report(f"Tight-window correlation dataset: {len(tightCorrDf):,} rows")
    report(tightCorrDf.groupby('Age')['corr'].agg(['count','mean','median','std']))
else:
    report("Tight-window correlation dataset is empty — no valid infant x parcel rows found.")


In [ ]:
# --- Plot: tight-window correlations by age ---
if not tightCorrDf.empty:

    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    bins = np.linspace(-1, 1, 41)
    for age, colour in [(6, 'crimson'), (12, 'firebrick')]:
        vals = tightCorrDf[tightCorrDf['Age'] == age]['corr'].dropna()
        ax.hist(vals, bins=bins, color=colour, alpha=0.55, edgecolor='white',
                label=f'{age}mo  (n={len(vals)}, median r={vals.median():.2f})')
    ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
    ax.set_xlabel(f'Fam1-{LABEL_B} waveform correlation (r), tight window 5-13s')
    ax.set_ylabel('Count')
    ax.set_title(f'1 v. Tight-window Fam1-{LABEL_B} waveform correlation')
    ax.legend(fontsize=9, frameon=False)
    fig.tight_layout(); plt.show()

    c6  = tightCorrDf[tightCorrDf['Age'] == 6]['corr'].dropna()
    c12 = tightCorrDf[tightCorrDf['Age'] == 12]['corr'].dropna()
    u = mannwhitneyu(c6, c12, alternative='greater')
    report(f"Mann-Whitney U (6mo > 12mo): U={u.statistic:.1f}, p={u.pvalue:.4g}")
    report(f"  6mo:  n={len(c6)},  median r = {c6.median():.3f}")
    report(f"  12mo: n={len(c12)}, median r = {c12.median():.3f}")

    # Compare to the wide-window results from 1a iii
    if 'corrDf' in dir():
        report("\n  Comparison with 1a iii (wide window 0-15s):")
        w6  = corrDf[corrDf['Age'] == 6]['corr'].dropna().median()
        w12 = corrDf[corrDf['Age'] == 12]['corr'].dropna().median()
        report(f"    6mo:  wide-window median r = {w6:.3f}, tight-window median r = {c6.median():.3f}")
        report(f"    12mo: wide-window median r = {w12:.3f}, tight-window median r = {c12.median():.3f}")


### 1 vi. Lag-optimised Fam1-Contrast Epoch correlation

If the contrast epoch is a shifted version of Fam1 (e.g. peaks at a different time), the zero-lag Pearson correlation underestimates their shape similarity. This test computes the correlation at multiple time lags between Fam1 and the contrast epoch (over the tight response window) and reports the **best (maximum) r** together with the **lag at which it was achieved**.

**Interpretation:**
- **Timing shift with preserved shape:** `max_r` recovers to high values; `best_lag` distribution is systematically nonzero at 12mo but centred on 0 at 6mo.
- **Genuine shape mismatch:** `max_r` stays low regardless of lag.
- **Genuine similarity:** `max_r` is high; `best_lag` ~= 0 at both ages.

*Caveat:* allowing lag-optimisation inflates the achievable r just by chance (lag chasing), so we compare max_r *distributions* between ages rather than against 0. `best_lag` distributions tell us whether observed shifts cluster (real) or scatter uniformly (chance).


In [ ]:
# --- 1 vi. Lag-optimised Fam1-contrast correlation ---
# For each pair, compute correlation at lags -30 to +30 samples (-3 to +3 s at 10 Hz)
# and keep the maximum r and the lag at which it was reached.
MAX_LAG_SAMPLES = 30
LAG_RANGE = np.arange(-MAX_LAG_SAMPLES, MAX_LAG_SAMPLES + 1)

def _lag_corr(fam1, fam3, base_window=TIGHT_CORR_WINDOW, lag_range=LAG_RANGE):
    """Return (max_r, best_lag_samples) after shifting the contrast epoch by each lag.
    Positive lag means the contrast epoch is delayed relative to fam1."""
    if fam1 is None or fam3 is None:
        return np.nan, np.nan
    rs = []
    for lag in lag_range:
        # slide fam3 by 'lag' samples: correlate fam1[t] with fam3[t + lag]
        # base_window defines the fam1 side; shift fam3's window accordingly
        f3_window = slice(base_window.start + lag, base_window.stop + lag)
        # skip if window goes out of bounds
        if f3_window.start < 0 or f3_window.stop > len(fam3):
            rs.append(np.nan)
            continue
        r = _pearson(fam1[base_window], fam3[f3_window])
        rs.append(r)
    rs = np.array(rs)
    if np.all(np.isnan(rs)):
        return np.nan, np.nan
    best_idx = int(np.nanargmax(rs))
    return float(rs[best_idx]), int(lag_range[best_idx])


lag_corr_rows = []
for age in (6, 12):
    age_meta = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Chrom'] == CHROM)]
    for parcel in target_parcels:
        parcel_meta = age_meta[age_meta['Parcel'] == parcel]
        for infant, inf_df in parcel_meta.groupby('Infant'):
            fam1 = _infant_trial_mean(inf_df, GROUP_A_TRIALS, tsDataset)
            fam3 = _infant_trial_mean(inf_df, GROUP_B_TRIALS, tsDataset)
            max_r, best_lag = _lag_corr(fam1, fam3)
            if np.isnan(max_r):
                continue
            lag_corr_rows.append({
                'Age': age, 'Parcel': parcel, 'Infant': infant,
                'max_r': max_r,
                'best_lag_samples': best_lag,
                'best_lag_seconds': best_lag / FS_HZ,
            })

lagCorrDf = pd.DataFrame(lag_corr_rows)
if not np.size(lagCorrDf) == 0:
    report(f"Lag-corr dataset: {len(lagCorrDf):,} rows")
    report("\n=== max_r ===")
    report(lagCorrDf.groupby('Age')['max_r'].agg(['count','mean','median','std']))
    report("\n=== best_lag (seconds; positive => contrast epoch delayed vs Fam1) ===")
    report(lagCorrDf.groupby('Age')['best_lag_seconds'].agg(['count','mean','median','std']))
else:
    report("Lag-corr dataset is empty — no valid infant x parcel rows found.")

In [ ]:
# --- Plot: max_r and best_lag distributions by age ---
if not lagCorrDf.empty:

    fig, axes = plt.subplots(2, 2, figsize=(11, 6))

    # --- Row 1: max_r distributions ---
    for col_idx, (age, colour) in enumerate([(6, 'crimson'), (12, 'firebrick')]):
        ax = axes[0, col_idx]
        vals = lagCorrDf[lagCorrDf['Age'] == age]['max_r'].dropna()
        bins = np.linspace(-1, 1, 41)
        ax.hist(vals, bins=bins, color=colour, alpha=0.7, edgecolor='white')
        ax.axvline(vals.median(), color='k', ls='-', lw=1.5,
                    label=f'median={vals.median():.2f}')
        ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
        ax.set_xlabel('max r across lags')
        ax.set_title(f'{age}mo: max r (n={len(vals)})', fontsize=10)
        ax.legend(fontsize=9, frameon=False)
    axes[0, 0].set_ylabel('Count')

    # --- Row 2: best_lag distributions ---
    for col_idx, (age, colour) in enumerate([(6, 'crimson'), (12, 'firebrick')]):
        ax = axes[1, col_idx]
        vals = lagCorrDf[lagCorrDf['Age'] == age]['best_lag_seconds'].dropna()
        bins = np.linspace(-3.5, 3.5, 30)
        ax.hist(vals, bins=bins, color=colour, alpha=0.7, edgecolor='white')
        ax.axvline(0, color='k', ls='--', lw=1, alpha=0.7)
        ax.axvline(vals.median(), color='k', ls='-', lw=1.5,
                    label=f'median={vals.median():.2f}s')
        ax.set_xlabel(f'best lag (s); +ve = {LABEL_B} delayed')
        ax.set_title(f'{age}mo: best lag (n={len(vals)})', fontsize=10)
        ax.legend(fontsize=9, frameon=False)
    axes[1, 0].set_ylabel('Count')

    fig.suptitle(f'1 vi. Lag-optimised Fam1-{LABEL_B} correlation: max r (top) and best lag (bottom)', y=1.00)
    fig.tight_layout(); plt.show()

    # --- Statistical comparisons ---
    r6  = lagCorrDf[lagCorrDf['Age'] == 6]['max_r'].dropna()
    r12 = lagCorrDf[lagCorrDf['Age'] == 12]['max_r'].dropna()
    u_r = mannwhitneyu(r6, r12, alternative='greater')
    report(f"Mann-Whitney U on max_r (H1: 6mo > 12mo): U={u_r.statistic:.1f}, p={u_r.pvalue:.4g}")
    report(f"  6mo:  median max_r = {r6.median():.3f}")
    report(f"  12mo: median max_r = {r12.median():.3f}")

    lag6  = lagCorrDf[lagCorrDf['Age'] == 6]['best_lag_seconds'].dropna()
    lag12 = lagCorrDf[lagCorrDf['Age'] == 12]['best_lag_seconds'].dropna()

    # Wilcoxon vs 0 for the shift being different from 0 at each age
    if len(lag6) > 0:
        w6  = wilcoxon(lag6)
        report(f"6mo:  Wilcoxon best_lag vs 0, p={w6.pvalue:.4g}, median = {lag6.median():.2f}s")
    if len(lag12) > 0:
        w12 = wilcoxon(lag12)
        report(f"12mo: Wilcoxon best_lag vs 0, p={w12.pvalue:.4g}, median = {lag12.median():.2f}s")
    u_lag = mannwhitneyu(lag12, lag6, alternative='greater')
    report(f"Mann-Whitney U on best_lag (H1: 12mo > 6mo lag): U={u_lag.statistic:.1f}, p={u_lag.pvalue:.4g}")


## 2. Within-infant vs between-infant response variability

The 1 i-vi results told us that per-infant Fam1 and contrast epoch responses share overall shape (max_r ~ 0.75 with lag) but with jittered timing, at BOTH ages. What we still don't know is where that jitter lives:

- **Within-infant jitter (attention-driven):** the same infant's response varies from trial to trial. Their trial-1 Fam1 peaks at 8s, their trial-3 Fam1 peaks at 10s. When you average trials to get their "mean Fam1", you get a smeared estimate of a noisy underlying response.
- **Between-infant jitter (individuation-driven):** each infant's own response is internally consistent trial-to-trial, but *different* infants have *different* characteristic timings. Infant A peaks at 8s reliably; infant B peaks at 11s reliably.

Both produce the same "the group average doesn't cleanly show a peak" signal, but they come from opposite mechanisms.

### Test

For each (Infant, Age, Parcel, Chrom=HbR), compute the response *per trial* (not averaged):
- **Within-infant consistency:** mean pairwise Pearson correlation across the 5 Fam1 trials (10 pairs) - and separately the 5 contrast epoch trials. High = infant is trial-to-trial consistent. Low = attentionally variable.
- **Between-infant agreement:** for each (Age, Parcel), compute the mean pairwise correlation of infants' trial-averaged Fam1 waveforms. High = infants agree on the response shape. Low = each infant has their own idiosyncratic pattern.

Predictions:
- **Attention-driven story:** within-infant consistency LOWER at 12mo than 6mo; between-infant agreement similar.
- **Individuation-driven story:** within-infant consistency similar; between-infant agreement LOWER at 12mo.
- **Both:** both lower at 12mo.
- **Neither:** no age differences in either measure -> would deepen the puzzle.


In [ ]:
# --- 2. Within-infant trial-to-trial consistency ---
# For each (Infant, Age, Parcel), compute mean pairwise correlation across
# trials within Fam1, and separately across trials within the contrast epoch.

def _get_trial_waveforms(inf_meta_df, epoch_trials, ts_all, target_len=231):
    """Return list of (trial_number, waveform) for valid trials in this epoch."""
    trials_out = []
    epoch_meta = inf_meta_df[inf_meta_df['Trial'].isin(epoch_trials)]
    for _, row in epoch_meta.iterrows():
        ts = ts_all[row['tsIDx']] * 1e6
        if np.isnan(ts).all():
            continue
        if len(ts) > target_len:
            ts = ts[:target_len]
        elif len(ts) < target_len:
            ts = np.pad(ts, (0, target_len - len(ts)), constant_values=np.nan)
        trials_out.append((row['Trial'], ts))
    return trials_out


def _mean_pairwise_corr(traces, window=slice(90, 170)):
    """Mean of all pairwise Pearson r among a list of traces in the given window."""
    if len(traces) < 2:
        return np.nan
    rs = []
    for i in range(len(traces)):
        for j in range(i + 1, len(traces)):
            r = _pearson(traces[i][window], traces[j][window])
            if not np.isnan(r):
                rs.append(r)
    return float(np.mean(rs)) if rs else np.nan


within_rows = []
for age in (6, 12):
    age_meta = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Chrom'] == CHROM)]
    for parcel in target_parcels:
        parcel_meta = age_meta[age_meta['Parcel'] == parcel]
        for infant, inf_df in parcel_meta.groupby('Infant'):
            fam1_trials = _get_trial_waveforms(inf_df, GROUP_A_TRIALS, tsDataset)
            fam3_trials = _get_trial_waveforms(inf_df, GROUP_B_TRIALS, tsDataset)
            fam1_traces = [t for _, t in fam1_trials]
            fam3_traces = [t for _, t in fam3_trials]

            r_fam1 = _mean_pairwise_corr(fam1_traces)
            r_fam3 = _mean_pairwise_corr(fam3_traces)
            within_rows.append({
                'Age': age, 'Parcel': parcel, 'Infant': infant,
                'within_r_Fam1': r_fam1,
                f'within_r_{LABEL_B}': r_fam3,
                'n_fam1_trials': len(fam1_trials),
                f'n_{LABEL_B}_trials': len(fam3_trials),
            })

withinDf = pd.DataFrame(within_rows)
if not np.size(withinDf) == 0:
    report(f"Within-infant dataset: {len(withinDf):,} rows")
    report("\n=== Within-infant consistency (mean pairwise r across 5 trials) ===")
    report(withinDf.groupby('Age')[['within_r_Fam1', f'within_r_{LABEL_B}']].describe().T)
else:
    report("Within-infant dataset is empty — no valid infant x parcel rows found.")


In [ ]:
# --- Plot: within-infant consistency by age ---
if not np.size(withinDf) == 0:

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    bins = np.linspace(-1, 1, 41)
    for col_idx, (age, colour_dark, colour_light) in enumerate(
        [(6, '#00008B', '#6495ED'), (12, '#00008B', '#6495ED')]
    ):
        ax = axes[col_idx]
        sub = withinDf[withinDf['Age'] == age]
        v1 = sub['within_r_Fam1'].dropna()
        v3 = sub[f'within_r_{LABEL_B}'].dropna()
        ax.hist(v1, bins=bins, color=colour_dark, alpha=0.55, edgecolor='white',
                label=f'Fam1  (n={len(v1)}, median={v1.median():.2f})')
        ax.hist(v3, bins=bins, color=colour_light, alpha=0.55, edgecolor='white',
                label=f'{LABEL_B}  (n={len(v3)}, median={v3.median():.2f})')
        ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
        ax.set_xlabel('Mean pairwise r across trials')
        ax.set_title(f'{age}mo: within-infant trial-to-trial consistency', fontsize=10)
        ax.legend(fontsize=8, frameon=False)
    axes[0].set_ylabel('Count')
    fig.suptitle('2 (i). Within-infant response consistency (per Infant x Parcel)', y=1.02)
    fig.tight_layout(); plt.show()

    # Statistical comparisons: 6mo > 12mo?
    for col in ('within_r_Fam1', f'within_r_{LABEL_B}'):
        v6  = withinDf[withinDf['Age'] == 6][col].dropna()
        v12 = withinDf[withinDf['Age'] == 12][col].dropna()
        u = mannwhitneyu(v6, v12, alternative='greater')
        report(f"{col}: Mann-Whitney U (H1: 6mo > 12mo) U={u.statistic:.1f}, p={u.pvalue:.4g}")
        report(f"  6mo:  median = {v6.median():.3f}")
        report(f"  12mo: median = {v12.median():.3f}")


In [ ]:
# --- 2. Between-infant agreement ---
# For each (Age, Parcel), compute pairwise correlations between all infants'
# trial-averaged Fam1 waveforms; then the same for the contrast epoch. Also compute the
# overall (Age, Parcel) mean pairwise r as a summary.

def _infant_mean_waveform(inf_meta_df, epoch_trials, ts_all, target_len=231):
    """Return the mean waveform across this infant's trials in this epoch, or None."""
    traces = [t for _, t in _get_trial_waveforms(inf_meta_df, epoch_trials, ts_all, target_len)]
    if not traces:
        return None
    return np.nanmean(traces, axis=0)


def _mean_pairwise_corr_of_infants(infant_means, window=slice(90, 170)):
    """Return list of all pairwise Pearson r's between infants' mean waveforms."""
    valid = [w for w in infant_means if w is not None]
    if len(valid) < 2:
        return []
    rs = []
    for i in range(len(valid)):
        for j in range(i + 1, len(valid)):
            r = _pearson(valid[i][window], valid[j][window])
            if not np.isnan(r):
                rs.append(r)
    return rs


between_rows = []
between_summary_rows = []

for age in (6, 12):
    age_meta = metadataDf[(metadataDf['Age'] == age) & (metadataDf['Chrom'] == CHROM)]
    for parcel in target_parcels:
        parcel_meta = age_meta[age_meta['Parcel'] == parcel]

        # For each infant at this parcel, compute their Fam1 and Contrast Epoch mean waveforms
        fam1_means, fam3_means = [], []
        for infant, inf_df in parcel_meta.groupby('Infant'):
            fam1_means.append(_infant_mean_waveform(inf_df, GROUP_A_TRIALS, tsDataset))
            fam3_means.append(_infant_mean_waveform(inf_df, GROUP_B_TRIALS, tsDataset))

        # Pairwise correlations across infants for this (age, parcel)
        rs_fam1 = _mean_pairwise_corr_of_infants(fam1_means)
        rs_fam3 = _mean_pairwise_corr_of_infants(fam3_means)

        # Store all pairwise values (for distribution plotting) with the parcel tag
        for r in rs_fam1:
            between_rows.append({'Age': age, 'Parcel': parcel, 'epoch': 'Fam1', 'r': r})
        for r in rs_fam3:
            between_rows.append({'Age': age, 'Parcel': parcel, 'epoch': LABEL_B, 'r': r})

        # Per-parcel summary: mean pairwise r
        between_summary_rows.append({
            'Age': age, 'Parcel': parcel,
            'between_mean_r_Fam1': float(np.mean(rs_fam1)) if rs_fam1 else np.nan,
            f'between_mean_r_{LABEL_B}': float(np.mean(rs_fam3)) if rs_fam3 else np.nan,
            'n_pairs_Fam1': len(rs_fam1),
            f'n_pairs_{LABEL_B}': len(rs_fam3),
        })

betweenDf = pd.DataFrame(between_rows)
betweenSummaryDf = pd.DataFrame(between_summary_rows)

if not np.size(betweenDf) == 0:
    report(f"Between-infant dataset: {len(betweenDf):,} pairwise r values")
    report(f"Between-infant summary: {len(betweenSummaryDf):,} (Age, Parcel) rows\n")

    report("=== Between-infant agreement per Age, Parcel (mean pairwise r) ===")
    report(betweenSummaryDf.groupby('Age')[['between_mean_r_Fam1', f'between_mean_r_{LABEL_B}']].describe().T)
else:
    report("Between-infant dataset is empty — no valid (Age, Parcel) pairs found.")


In [ ]:
# --- Plot: between-infant agreement by age (per parcel and pooled) ---
if not np.size(betweenSummaryDf) == 0:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    bins = np.linspace(-1, 1, 41)

    for col_idx, age in enumerate((6, 12)):
        ax = axes[col_idx]
        sub = betweenSummaryDf[betweenSummaryDf['Age'] == age]
        v1 = sub['between_mean_r_Fam1'].dropna()
        v3 = sub[f'between_mean_r_{LABEL_B}'].dropna()
        ax.hist(v1, bins=bins, color='#00008B', alpha=0.6, edgecolor='white',
                label=f'Fam1  (n_parcels={len(v1)}, median={v1.median():.2f})')
        ax.hist(v3, bins=bins, color='#6495ED', alpha=0.6, edgecolor='white',
                label=f'{LABEL_B}  (n_parcels={len(v3)}, median={v3.median():.2f})')
        ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
        ax.set_xlabel('Mean pairwise r between infants (per parcel)')
        ax.set_title(f'{age}mo: between-infant agreement', fontsize=10)
        ax.legend(fontsize=9, frameon=False)
    axes[0].set_ylabel('Count (parcels)')
    fig.suptitle('2 (ii). Between-infant response agreement (per parcel)', y=1.02)
    fig.tight_layout(); plt.show()

    # Comparisons at the parcel level
    for col in ('between_mean_r_Fam1', f'between_mean_r_{LABEL_B}'):
        v6  = betweenSummaryDf[betweenSummaryDf['Age'] == 6][col].dropna()
        v12 = betweenSummaryDf[betweenSummaryDf['Age'] == 12][col].dropna()
        u = mannwhitneyu(v6, v12, alternative='greater')
        report(f"{col}: Mann-Whitney U (H1: 6mo > 12mo across parcels) U={u.statistic:.1f}, p={u.pvalue:.4g}")
        report(f"  6mo:  n_parcels={len(v6)}, median = {v6.median():.3f}")
        report(f"  12mo: n_parcels={len(v12)}, median = {v12.median():.3f}")


In [ ]:
# --- Synthesis: which mechanism does the pattern support? ---
if not np.size(withinDf) == 0 and not np.size(betweenSummaryDf) == 0: 

    # Overall medians
    w6_f1  = withinDf[withinDf['Age'] == 6]['within_r_Fam1'].median()
    w12_f1 = withinDf[withinDf['Age'] == 12]['within_r_Fam1'].median()
    w6_f3  = withinDf[withinDf['Age'] == 6][f'within_r_{LABEL_B}'].median()
    w12_f3 = withinDf[withinDf['Age'] == 12][f'within_r_{LABEL_B}'].median()

    b6_f1  = betweenSummaryDf[betweenSummaryDf['Age'] == 6]['between_mean_r_Fam1'].median()
    b12_f1 = betweenSummaryDf[betweenSummaryDf['Age'] == 12]['between_mean_r_Fam1'].median()
    b6_f3  = betweenSummaryDf[betweenSummaryDf['Age'] == 6][f'between_mean_r_{LABEL_B}'].median()
    b12_f3 = betweenSummaryDf[betweenSummaryDf['Age'] == 12][f'between_mean_r_{LABEL_B}'].median()

    report("\nSummary of median correlations by age:\n")
    report(f"                        6mo      12mo    diff(6-12)")
    report(f"  Within-infant Fam1:   {w6_f1:5.3f}   {w12_f1:5.3f}   {w6_f1 - w12_f1:+.3f}")
    report(f"  Within-infant {LABEL_B}:   {w6_f3:5.3f}   {w12_f3:5.3f}   {w6_f3 - w12_f3:+.3f}")
    report(f"  Between-infant Fam1:  {b6_f1:5.3f}   {b12_f1:5.3f}   {b6_f1 - b12_f1:+.3f}")
    report(f"  Between-infant {LABEL_B}:  {b6_f3:5.3f}   {b12_f3:5.3f}   {b6_f3 - b12_f3:+.3f}")

    report('\nInterpretation guide:')
    report('  Within-infant drops with age  -> attention/state variability')
    report('  Between-infant drops with age -> individuation/specialisation')
    report('  Both drop                     -> both mechanisms')
    report('  Neither drops                 -> neither; deepens the puzzle')


# Section 3 — personal-ROI follow-up

Sections 1 and 2 concluded that per-parcel per-infant tests are dominated by
noise at these SNRs. Section 3 rebuilds the same analyses at an
**infant-specific ROI** (the set of parcels where that infant showed a
significant activation response), which pools signal across parcels and should
substantially improve individual-level SNR.

### ROI selection: switchable threshold

Change `ROI_THRESHOLD` at the top of section 3.1 to switch between two ways of
defining an infant's activated parcels:

- `'fdr'`         -> activation-significant at the individual-level FDR-corrected threshold
                   (`resultsDf['Significant'] == True`)
- `'uncorrected'` -> activation-significant at uncorrected p < 0.05
                   (`resultsDf['p at peak (point)'] < 0.05`)

Re-run everything from 3.1 downward after switching.


## 3.1 Build personal ROIs at both thresholds

In [ ]:
# --- 3.1 Personal ROI selection ---
# Decide whether to plot meshes or not (computationally expensive)
PLOT_MESHES = 'yes'                  # 'yes' or 'no'

# Load activation results (from the individual-level TFCE pipeline).
RESULTS_CSV = DATA + "surf_parcel_TFCE_circshift_results_INDIV_spatiotemporal_FDR.csv"
resultsDf = pd.read_csv(RESULTS_CSV)

# Filter by chromophore (chrom only) and by the chosen significance criterion.
res_chrom = resultsDf[resultsDf['Chrom'] == CHROM].copy()
if ROI_THRESHOLD == 'fdr':
    roi_rows = res_chrom[res_chrom['Significant'] == True]
elif ROI_THRESHOLD == 'uncorrected':
    if 'p at peak (point)' in res_chrom.columns:
        roi_rows = res_chrom[res_chrom['p at peak (point)'] < 0.05]
    else:
        raise ValueError("resultsDf doesn't have 'p at peak (point)' column; "
                          "can't apply uncorrected threshold.")
else:
    raise ValueError(f"Unknown ROI_THRESHOLD: {ROI_THRESHOLD}")

# Build the per-(Age, Infant) list of ROI parcels.
personal_rois = {}
for (age, infant), grp in roi_rows.groupby(['Age', 'Infant']):
    personal_rois[(age, infant)] = sorted(grp['Parcel'].unique().tolist())

# Report basic counts per age
report(f"ROI threshold: '{ROI_THRESHOLD}'")
report(f"Contrast: '{HAB_CONTRAST}' (Fam1 vs {LABEL_B})")
report(f"Total (Age, Infant) combinations with any activated parcels: {len(personal_rois):,}\n")
for age in (6, 12):
    infants_with_roi = [inf for (a, inf) in personal_rois if a == age]
    sizes = [len(personal_rois[(age, inf)]) for inf in infants_with_roi]
    report(f"  {age}mo: {len(infants_with_roi)} infants have any activated parcels")
    if sizes:
        report(f"    ROI size: median = {np.median(sizes):.0f}, mean = {np.mean(sizes):.1f}, "
              f"min = {min(sizes)}, max = {max(sizes)}")


## 3.2 Compute per-infant ROI HRFs

For each (Age, Infant) with a non-empty ROI, compute their Fam1 and Contrast Epoch mean
waveforms across trials and across ROI parcels (averaging across parcels first,
then across trials). Store also per-trial ROI waveforms for section 3.9 (within-
infant variability).

In [ ]:
# --- 3.2 Per-infant ROI HRF construction ---
# For each (Age, Infant), get their ROI HRF for Fam1 and Contrast Epoch, plus the per-trial
# waveforms so we can compute within-infant consistency later.

# HbR metadata restricted to target ages
_meta_chrom = metadataDf[(metadataDf['Chrom'] == CHROM)]

roi_hrfs = {}          # (age, infant) -> {'Fam1': mean_trace, LABEL_B: mean_trace,
                       #                   'Fam1_trials': [traces], f'{LABEL_B}_trials': [traces],
                       #                   'roi_parcels': [...], 'n_Fam1_trials_valid': int,
                       #                   f'n_{LABEL_B}_trials_valid': int}

for (age, infant), roi_parcels in personal_rois.items():
    if not roi_parcels:
        continue
    inf_meta = _meta_chrom[(_meta_chrom['Age'] == age) & (_meta_chrom['Infant'] == infant)
                          & (_meta_chrom['Parcel'].isin(roi_parcels))]
    if inf_meta.empty:
        continue

    # Group by Trial, then average across ROI parcels within each trial
    trial_to_roi_trace = {}
    for trial, trial_meta in inf_meta.groupby('Trial'):
        parcel_traces = []
        for _, row in trial_meta.iterrows():
            ts = tsDataset[row['tsIDx']] * 1e6
            if np.isnan(ts).all():
                continue
            if len(ts) > 231:
                ts = ts[:231]
            elif len(ts) < 231:
                ts = np.pad(ts, (0, 231 - len(ts)), constant_values=np.nan)
            parcel_traces.append(ts)
        if parcel_traces:
            trial_to_roi_trace[trial] = np.nanmean(parcel_traces, axis=0)

    # Split by epoch
    fam1_trial_traces = [trial_to_roi_trace[t] for t in GROUP_A_TRIALS if t in trial_to_roi_trace]
    fam3_trial_traces = [trial_to_roi_trace[t] for t in GROUP_B_TRIALS if t in trial_to_roi_trace]

    if not fam1_trial_traces or not fam3_trial_traces:
        continue

    roi_hrfs[(age, infant)] = {
        'Fam1': np.nanmean(fam1_trial_traces, axis=0),
        LABEL_B: np.nanmean(fam3_trial_traces, axis=0),
        'Fam1_trials': fam1_trial_traces,
        f'{LABEL_B}_trials': fam3_trial_traces,
        'roi_parcels': roi_parcels,
        'n_Fam1_trials_valid': len(fam1_trial_traces),
        f'n_{LABEL_B}_trials_valid': len(fam3_trial_traces),
    }

report(f"Built ROI HRFs for {len(roi_hrfs)} (Age, Infant) combinations.")
for age in (6, 12):
    n = sum(1 for (a, _) in roi_hrfs if a == age)
    report(f"  {age}mo: {n} infants")


## 3.3 ROI-level data quality diagnostic (analog of 1 iv)

Does averaging across the ROI parcels actually gain us SNR? If ROI SNR is
substantially higher than parcel SNR (~3 at both ages from 1 iv), then downstream
ROI-level tests are much more informative than their parcel-level counterparts.
If not, we've hit a fundamental noise floor and the personal-ROI approach doesn't
help.

In [ ]:
# --- 3.3 ROI-level SNR ---

def _snr_of_mean(trace, baseline_slice=slice(0, 40), peak_slice=slice(120, 160)):
    if trace is None or np.all(np.isnan(trace)):
        return np.nan
    baseline_seg = trace[baseline_slice]
    peak_seg = trace[peak_slice]
    if np.all(np.isnan(baseline_seg)) or np.all(np.isnan(peak_seg)):
        return np.nan
    baseline_sd = np.nanstd(baseline_seg)
    peak_amp = np.nanmax(np.abs(peak_seg - np.nanmean(baseline_seg)))
    return peak_amp / baseline_sd if baseline_sd > 0 else np.nan


roi_diag_rows = []
for (age, infant), h in roi_hrfs.items():
    roi_diag_rows.append({
        'Age': age, 'Infant': infant,
        'roi_size': len(h['roi_parcels']),
        'n_Fam1_trials_valid': h['n_Fam1_trials_valid'],
        f'n_{LABEL_B}_trials_valid': h[f'n_{LABEL_B}_trials_valid'],
        'snr_Fam1': _snr_of_mean(h['Fam1']),
        f'snr_{LABEL_B}': _snr_of_mean(h[LABEL_B]),
    })

roiDiagDf = pd.DataFrame(roi_diag_rows)
report(f"ROI diagnostic dataset: {len(roiDiagDf):,} rows")
report("\n=== ROI-level SNR ===")
report(roiDiagDf.groupby('Age')[['snr_Fam1', f'snr_{LABEL_B}']].describe().T)
report("\n=== Comparison with parcel-level (from 1 iv) ===")
report("  Parcel-level median SNR was ~3.1-6.1 at both ages.")


report("\n=== Between-age SNR comparison (Mann-Whitney U, two-sided) ===")
for snr_col in ['snr_Fam1', f'snr_{LABEL_B}']:
    v6  = roiDiagDf[roiDiagDf['Age'] == 6][snr_col].dropna()
    v12 = roiDiagDf[roiDiagDf['Age'] == 12][snr_col].dropna()
    if len(v6) > 0 and len(v12) > 0:
        u = mannwhitneyu(v6, v12, alternative='two-sided')
        report(f"  {snr_col}: 6mo median = {v6.median():.3f} (n={len(v6)}), "
              f"12mo median = {v12.median():.3f} (n={len(v12)}), "
              f"U = {u.statistic:.1f}, p = {u.pvalue:.4g}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
bins = np.linspace(0, 30, 40)
for col_idx, age in enumerate((6, 12)):
    sub = roiDiagDf[roiDiagDf['Age'] == age]
    ax = axes[col_idx]
    ax.hist([sub['snr_Fam1'].dropna(), sub[f'snr_{LABEL_B}'].dropna()],
            bins=bins, label=['Fam1', 'Fam3'],
            color=['#00008B', '#6495ED'], alpha=0.75, edgecolor='white')
    ax.axvline(3, color='k', ls='--', lw=1, alpha=0.7, label='SNR=3 (parcel-level ~median)')
    ax.set_title(f'{age}mo: ROI-level SNR', fontsize=10)
    ax.set_xlabel('SNR (peak / baseline SD)')
    ax.set_ylabel('Count')
    ax.legend(frameon=False, fontsize=8)
fig.suptitle('3.3. ROI-level data-quality diagnostic', y=1.02)
fig.tight_layout(); plt.show()


## 3.4 ROI-level group-averaged HRFs, 6mo vs 12mo (analog of 1)

The load-bearing plot. Each infant contributes their personal-ROI HRF (Fam1 and
the contrast epoch, HbR). We average across infants within age. Since the ROI is defined by
individual activation, each infant contributes from wherever their own response
lives — the group average shouldn't be confounded by a fixed anatomical mask.

In [ ]:
# --- 3.4 Group ROI-HRF plot ---

def _stack_hrfs(age, epoch):
    curves = [roi_hrfs[(a, inf)][epoch] for (a, inf) in roi_hrfs if a == age]
    if not curves:
        return None, None, 0
    stack = np.vstack(curves)
    mean_trace = np.nanmean(stack, axis=0)
    sem_trace = np.nanstd(stack, axis=0) / np.sqrt(stack.shape[0])
    return mean_trace, sem_trace, stack.shape[0]


fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for col_idx, age in enumerate((6, 12)):
    ax = axes[col_idx]
    for epoch in ('Fam1', LABEL_B):
        mean_trace, sem_trace, n = _stack_hrfs(age, epoch)
        if mean_trace is None:
            continue
        s = _style(CHROM, epoch)   # styling from section 1
        ax.plot(TIME_AXIS, mean_trace, label=f"{epoch} (n={n})", lw=1.8, **s)
        ax.fill_between(TIME_AXIS, mean_trace - sem_trace, mean_trace + sem_trace,
                        color=s['color'], alpha=0.2, lw=0)
    ax.axhline(0, color='grey', lw=0.5, alpha=0.7)
    ax.axvline(0, color='grey', lw=0.5, ls=':', alpha=0.7)
    ax.axvspan((PEAK_ONSET - STIM_ONSET) / FS_HZ,
               (PEAK_OFFSET - STIM_ONSET) / FS_HZ, color='yellow', alpha=0.10, lw=0)
    ax.set_title(f'{age}mo — mean of personal-ROI HRFs', fontsize=10)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=True, bottom=True)
    ax.set_xlabel('Time (s)')
    ax.legend(loc='upper right', fontsize=9, frameon=False)
axes[0].set_ylabel(f'$\\Delta$ {CHROM_LABELS[CHROM]} ($\\mu$M)')
fig.suptitle(f"3.4. Group-averaged personal-ROI HRFs (threshold: '{ROI_THRESHOLD}', {CHROM_LABELS[CHROM]}, {HAB_CONTRAST})", y=1.02)
fig.tight_layout(); plt.show()


In [ ]:
# --- 3.4b All-contrast ROI-level group-averaged HRFs (Fam1, Fam3, Novel, Post) ---

ALL_EPOCH_TRIALS = {
    'Fam1':  list(range(1, 6)),
    'Fam3':  list(range(11, 16)),
    'Novel': list(range(16, 21)),
    'Post':  list(range(21, 26)),
}
EPOCH_ORDER = ['Fam1', 'Fam3', 'Novel', 'Post']

roi_hrfs_all = {}   # (age, infant) -> {epoch: mean_trace}

for (age, infant), roi_parcels in personal_rois.items():
    if not roi_parcels:
        continue
    inf_meta = _meta_chrom[(_meta_chrom['Age'] == age) & (_meta_chrom['Infant'] == infant)
                          & (_meta_chrom['Parcel'].isin(roi_parcels))]
    if inf_meta.empty:
        continue

    # Same per-trial ROI averaging as 3.2, just not restricted to group_a/group_b.
    trial_to_roi_trace = {}
    for trial, trial_meta in inf_meta.groupby('Trial'):
        parcel_traces = []
        for _, row in trial_meta.iterrows():
            ts = tsDataset[row['tsIDx']] * 1e6
            if np.isnan(ts).all():
                continue
            if len(ts) > 231:
                ts = ts[:231]
            elif len(ts) < 231:
                ts = np.pad(ts, (0, 231 - len(ts)), constant_values=np.nan)
            parcel_traces.append(ts)
        if parcel_traces:
            trial_to_roi_trace[trial] = np.nanmean(parcel_traces, axis=0)

    epoch_traces = {}
    for epoch, trials in ALL_EPOCH_TRIALS.items():
        traces = [trial_to_roi_trace[t] for t in trials if t in trial_to_roi_trace]
        if traces:
            epoch_traces[epoch] = np.nanmean(traces, axis=0)

    # Keep only infants with all four epochs, so all four curves at a given age
    # are averaged over the same set of infants.
    if all(e in epoch_traces for e in EPOCH_ORDER):
        roi_hrfs_all[(age, infant)] = epoch_traces

report(f"Built all-contrast ROI HRFs for {len(roi_hrfs_all)} (Age, Infant) combinations "
      f"(infants with valid Fam1, Fam3, Novel, and Post trials).")
for age in (6, 12):
    n = sum(1 for (a, _) in roi_hrfs_all if a == age)
    report(f"  {age}mo: {n} infants")


# --- Styling: Fam1 darkest + solid; Fam3/Novel/Post progressively lighter, ---
# --- each with a distinct dash pattern (dashed / dotted / dash-dot).       ---
ALL_EPOCH_STYLES = {
    'HbO': {
        'Fam1':  {'color': '#8B0000', 'linestyle': '-'},
        'Fam3':  {'color': '#CD5C5C', 'linestyle': '--'},
        'Novel': {'color': '#FF6961', 'linestyle': ':'},
        'Post':  {'color': '#FFB3AD', 'linestyle': '-.'},
    },
    'HbR': {
        'Fam1':  {'color': '#00008B', 'linestyle': '-'},
        'Fam3':  {'color': '#4169E1', 'linestyle': '--'},
        'Novel': {'color': '#6495ED', 'linestyle': ':'},
        'Post':  {'color': '#A9C6FF', 'linestyle': '-.'},
    },
}

def _style_all(chrom, epoch):
    label = CHROM_LABELS.get(chrom, 'HbO')
    return ALL_EPOCH_STYLES.get(label, ALL_EPOCH_STYLES['HbO']).get(
        epoch, {'color': 'black', 'linestyle': '-'})


def _stack_hrfs_all(age, epoch):
    curves = [roi_hrfs_all[(a, inf)][epoch] for (a, inf) in roi_hrfs_all if a == age]
    if not curves:
        return None, None, 0
    stack = np.vstack(curves)
    mean_trace = np.nanmean(stack, axis=0)
    sem_trace = np.nanstd(stack, axis=0) / np.sqrt(stack.shape[0])
    return mean_trace, sem_trace, stack.shape[0]


# Shared y-limits computed across BOTH age panels and all four epochs, so
# amplitudes are directly comparable across both figures.
y_vals = []
for age in (6, 12):
    for epoch in EPOCH_ORDER:
        mean_trace, sem_trace, n = _stack_hrfs_all(age, epoch)
        if mean_trace is None:
            continue
        y_vals.append(mean_trace + sem_trace)
        y_vals.append(mean_trace - sem_trace)
y_all = np.concatenate(y_vals) if y_vals else np.array([0.0])
y_lim = (np.nanmin(y_all) * 1.1, np.nanmax(y_all) * 1.1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for col_idx, age in enumerate((6, 12)):
    ax = axes[col_idx]
    for epoch in EPOCH_ORDER:
        mean_trace, sem_trace, n = _stack_hrfs_all(age, epoch)
        if mean_trace is None:
            continue
        s = _style_all(CHROM, epoch)
        ax.plot(TIME_AXIS, mean_trace, label=epoch, lw=1.8, **s)
        ax.fill_between(TIME_AXIS, mean_trace - sem_trace, mean_trace + sem_trace,
                        color=s['color'], alpha=0.15, lw=0)
    ax.axhline(0, color='grey', lw=0.5, alpha=0.7)
    ax.axvline(0, color='grey', lw=0.5, ls=':', alpha=0.7)
    ax.axvspan((PEAK_ONSET - STIM_ONSET) / FS_HZ,
               (PEAK_OFFSET - STIM_ONSET) / FS_HZ, color='yellow', alpha=0.10, lw=0)
    ax.set_ylim(y_lim)
    ax.set_title(f'{age}mo — mean of personal-ROI HRFs, all contrasts', fontsize=10)
    ax.set_xlabel('Time (s)')
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(left=True, bottom=True)
    ax.legend(loc='lower right', fontsize=9, frameon=False)
axes[0].set_ylabel(f'$\\Delta$ {CHROM_LABELS[CHROM]} ($\\mu$M)')
fig.suptitle(f"3.4b. Group-averaged personal-ROI HRFs — Fam1, Fam3, Novel, Post "
            f"(threshold: '{ROI_THRESHOLD}', {CHROM_LABELS[CHROM]})", y=1.02)
fig.tight_layout(); plt.show()

## 3.5 ROI size distribution per age

How many parcels does each infant activate? If 12mo infants have systematically
smaller (more focal) ROIs, that supports the individuation account; if larger,
that would suggest more diffuse responses at 12mo.

In [ ]:
# --- 3.5 ROI size ---

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
bins = np.linspace(0, roiDiagDf['roi_size'].max()+2, 40)
for age, colour in [(6, 'crimson'), (12, 'firebrick')]:
    sizes = roiDiagDf[roiDiagDf['Age'] == age]['roi_size']
    ax.hist(sizes, bins=bins, color=colour, alpha=0.6, edgecolor='white',
            label=f'{age}mo  (n={len(sizes)}, median={sizes.median():.0f})')
ax.set_xlabel('ROI size (number of parcels)')
ax.set_ylabel('Count (infants)')
ax.set_title(f"3.5. Personal-ROI size distribution per age (threshold: '{ROI_THRESHOLD}')")
ax.legend(fontsize=9, frameon=False)
fig.tight_layout(); plt.show()

s6  = roiDiagDf[roiDiagDf['Age'] == 6]['roi_size']
s12 = roiDiagDf[roiDiagDf['Age'] == 12]['roi_size']
u = mannwhitneyu(s6, s12, alternative='two-sided')
report(f"Mann-Whitney U (two-sided): U={u.statistic:.1f}, p={u.pvalue:.4g}")
report(f"  6mo:  median ROI size = {s6.median():.0f}")
report(f"  12mo: median ROI size = {s12.median():.0f}")


## 3.6 Habituation amplitude per infant

For each infant, compute the peak-window mean (8-12s post-stim) for Fam1 and
the contrast group in their ROI HRF. Habituation amplitude = Fam1 - contrast peak.
Positive values are canonical habituation (for HbO; for HbR the sign is flipped
upfront via a HAB_SIGN switch).

In [ ]:
# --- 3.6 Habituation amplitude ---

peak_slice = slice(PEAK_ONSET, PEAK_OFFSET)

hab_amp_rows = []
for (age, infant), h in roi_hrfs.items():
    fam1_pk = float(np.nanmean(h['Fam1'][peak_slice]))
    fam3_pk = float(np.nanmean(h[LABEL_B][peak_slice]))
    hab_amp_rows.append({
        'Age': age, 'Infant': infant,
        'Fam1_peak': fam1_pk, f'{LABEL_B}_peak': fam3_pk,
        'hab_amp': fam1_pk - fam3_pk,
    })

habAmpDf = pd.DataFrame(hab_amp_rows)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
bins = 30
for age, colour in [(6, 'crimson'), (12, 'firebrick')]:
    vals = habAmpDf[habAmpDf['Age'] == age]['hab_amp']
    ax.hist(vals, bins=bins, color=colour, alpha=0.6, edgecolor='white',
            label=f'{age}mo  (n={len(vals)}, median={vals.median():.3f})')
ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
ax.set_xlabel(f'Habituation amplitude (Fam1 - {LABEL_B}, $\\Delta$ {CHROM_LABELS[CHROM]} $\\mu$M)')
ax.set_ylabel('Count (infants)')
ax.set_title(f"3.6. Per-infant habituation amplitude (threshold: '{ROI_THRESHOLD}', {HAB_CONTRAST})")
ax.legend(fontsize=9, frameon=False)
fig.tight_layout(); plt.show()

# Wilcoxon signed-rank vs 0. For HbR, canonical habituation is a *negative*
# change (opposite sign convention to HbO), so H1 here is hab_amp < 0,
# tested via alternative='less' below.
for age in (6, 12):
    vals = habAmpDf[habAmpDf['Age'] == age]['hab_amp'].dropna()
    if len(vals) > 0:
        w = wilcoxon(vals, alternative='less')
        report(f"{age}mo: Wilcoxon vs 0 (H1: Fam1 < {LABEL_B}) - n={len(vals)}, "
              f"W={w.statistic:.1f}, p={w.pvalue:.4g}, median = {vals.median():.4f}")

# Mann-Whitney U comparing 6mo and 12mo (H1: 6mo > 12mo)
h6  = habAmpDf[habAmpDf['Age'] == 6]['hab_amp'].dropna()
h12 = habAmpDf[habAmpDf['Age'] == 12]['hab_amp'].dropna()
u = mannwhitneyu(h6, h12, alternative='greater')

report("Habituation amplitude (per-infant)")
report(f"\nMann-Whitney U (H1: 6mo > 12mo hab_amp): U={u.statistic:.1f}, p={u.pvalue:.4g}")


## 3.7 ROI-level Fam1-Contrast Epoch waveform correlation

Analog of 1 iii (wide window), 1 v (tight window), and 1 vi (lag-optimised). At
the ROI level, each infant contributes one Fam1 and one contrast epoch waveform. Correlate
them per infant. Compare distributions across ages.

**Predictions:**
- If SNR was the limiting factor at parcel level: 6mo correlations recover to
  r ~ 0.75+; 12mo either recovers similarly (magnitude habituation) or stays
  lower (shape difference).
- If both ages show high r: no shape difference between epochs at either age.
- If both stay low: something more fundamental about the response is preventing
  Fam1-contrast epoch alignment even at ROI SNR.

In [ ]:
# --- 3.7 ROI-level Fam1-Contrast Epoch correlations ---

def _pair_wave(fam1, fam3, window):
    return _pearson(fam1[window], fam3[window])


def _lag_corr(fam1, fam3, base_window=slice(90, 170), max_lag=30):
    lag_range = np.arange(-max_lag, max_lag + 1)
    rs = []
    for lag in lag_range:
        w = slice(base_window.start + lag, base_window.stop + lag)
        if w.start < 0 or w.stop > len(fam3):
            rs.append(np.nan); continue
        rs.append(_pearson(fam1[base_window], fam3[w]))
    rs = np.array(rs)
    if np.all(np.isnan(rs)):
        return np.nan, np.nan
    best = int(np.nanargmax(rs))
    return float(rs[best]), int(lag_range[best])


CORR_WIDE = slice(40, 190)     # 0-15s post-stim
CORR_TIGHT = slice(90, 170)    # 5-13s post-stim

roi_corr_rows = []
for (age, infant), h in roi_hrfs.items():
    r_wide  = _pair_wave(h['Fam1'], h[LABEL_B], CORR_WIDE)
    r_tight = _pair_wave(h['Fam1'], h[LABEL_B], CORR_TIGHT)
    r_lag, best_lag = _lag_corr(h['Fam1'], h[LABEL_B], CORR_TIGHT)
    roi_corr_rows.append({
        'Age': age, 'Infant': infant,
        'r_wide': r_wide, 'r_tight': r_tight,
        'r_lag_max': r_lag, 'best_lag_seconds': best_lag / FS_HZ if best_lag is not None else np.nan,
    })

roiCorrDf = pd.DataFrame(roi_corr_rows)
report(f"ROI correlation dataset: {len(roiCorrDf):,} rows")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
bins = np.linspace(-1, 1, 41)
titles = [('r_wide', 'Wide window (0-15s)'),
          ('r_tight', 'Tight window (5-13s)'),
          ('r_lag_max', 'Lag-optimised max r')]
for ax, (col, title) in zip(axes, titles):
    for age, colour in [(6, 'crimson'), (12, 'firebrick')]:
        vals = roiCorrDf[roiCorrDf['Age'] == age][col].dropna()
        ax.hist(vals, bins=bins, color=colour, alpha=0.55, edgecolor='white',
                label=f'{age}mo  (n={len(vals)}, median r={vals.median():.2f})')
    ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
    ax.set_xlabel(f'Fam1-{LABEL_B} r ({title})')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=8, frameon=False)
axes[0].set_ylabel('Count (infants)')
fig.suptitle(f"3.7. ROI-level Fam1-{LABEL_B} correlation (threshold: '{ROI_THRESHOLD}')", y=1.02)
fig.tight_layout(); plt.show()

# Statistical comparisons
for col, title in titles:
    v6  = roiCorrDf[roiCorrDf['Age'] == 6][col].dropna()
    v12 = roiCorrDf[roiCorrDf['Age'] == 12][col].dropna()
    if len(v6) > 0 and len(v12) > 0:
        u = mannwhitneyu(v6, v12, alternative='greater')
        report(f"{col:12s}: Mann-Whitney U (H1: 6mo > 12mo) U={u.statistic:.1f}, p={u.pvalue:.4g}, "
              f"medians 6mo={v6.median():.3f}, 12mo={v12.median():.3f}")


## 3.8 ROI-level latency (analog of 1 i)

For each infant, find the peak time (argmax within samples 40-170 = 0-13s
post-stim) of their ROI Fam1 and contrast epoch waveforms. The difference (contrast epoch - Fam1) is the per-infant latency shift.

**Prediction under the timing-shift interpretation:** at 12mo, contrast epoch peaks later
than Fam1 (positive difference); at 6mo, no shift.

In [ ]:
# --- 3.8 ROI-level latency ---
LATENCY_WINDOW = slice(40, 170)

latency_roi_rows = []
for (age, infant), h in roi_hrfs.items():
    f1w = h['Fam1'][LATENCY_WINDOW]; f3w = h[LABEL_B][LATENCY_WINDOW]
    if np.all(np.isnan(f1w)) or np.all(np.isnan(f3w)):
        continue
    f1_peak_idx = int(np.nanargmin(f1w))
    f3_peak_idx = int(np.nanargmin(f3w))
    f1_peak_s = (f1_peak_idx + LATENCY_WINDOW.start - STIM_ONSET) / FS_HZ
    f3_peak_s = (f3_peak_idx + LATENCY_WINDOW.start - STIM_ONSET) / FS_HZ
    latency_roi_rows.append({
        'Age': age, 'Infant': infant,
        'Fam1_peak_s': f1_peak_s, f'{LABEL_B}_peak_s': f3_peak_s,
        'diff_s': f3_peak_s - f1_peak_s,
    })

latencyRoiDf = pd.DataFrame(latency_roi_rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, age in zip(axes, (6, 12)):
    sub = latencyRoiDf[latencyRoiDf['Age'] == age]['diff_s']
    ax.hist(sub, bins=25, color=('crimson' if age == 6 else 'firebrick'),
            alpha=0.7, edgecolor='white')
    ax.axvline(0, color='k', ls='--', lw=1, alpha=0.7)
    ax.axvline(sub.median(), color='black', lw=1.5, label=f'median={sub.median():.2f}s')
    ax.set_title(f'{age}mo (n={len(sub)})', fontsize=10)
    ax.set_xlabel(f'{LABEL_B} peak time - Fam1 peak time (s)')
    ax.legend(fontsize=9, frameon=False)
axes[0].set_ylabel('Count (infants)')
fig.suptitle(f"3.8. ROI-level latency shift (threshold: '{ROI_THRESHOLD}')", y=1.02)
fig.tight_layout(); plt.show()

for age in (6, 12):
    diffs = latencyRoiDf[latencyRoiDf['Age'] == age]['diff_s'].dropna()
    if len(diffs) > 0:
        w = wilcoxon(diffs, alternative='greater')
        report(f"{age}mo: n={len(diffs)}, Wilcoxon (H1: {LABEL_B} later) W={w.statistic:.1f}, p={w.pvalue:.4g}, median = {diffs.median():.2f}s")


## 3.9 ROI-level within-vs-between infant variability (analog of 2)

At the ROI level:
- **Within-infant:** correlations across an infant's trial-level ROI waveforms
  (each trial averaged across ROI parcels; per epoch).
- **Between-infant:** correlations between infants' mean ROI waveforms (per
  epoch, at their own respective ROIs, since ROIs differ across infants).

Note: between-infant correlation at the ROI level compares each infant's *own*
ROI HRF to every other infant's *own* ROI HRF. This is different from the
per-parcel section 2 test which compared across infants at the same parcel.

In [ ]:
# --- 3.9 Within- and between-infant variability at ROI level ---

TIGHT = slice(90, 170)

# Within-infant: per (age, infant), mean pairwise r across trial-level ROI traces
within_roi_rows = []
for (age, infant), h in roi_hrfs.items():
    r_fam1 = _mean_pairwise_corr(h['Fam1_trials'], window=TIGHT)
    r_fam3 = _mean_pairwise_corr(h[f'{LABEL_B}_trials'], window=TIGHT)
    within_roi_rows.append({'Age': age, 'Infant': infant,
                             'within_r_Fam1': r_fam1, f'within_r_{LABEL_B}': r_fam3})

withinRoiDf = pd.DataFrame(within_roi_rows)
report(f"Within-infant ROI dataset: {len(withinRoiDf):,} rows")
report(withinRoiDf.groupby('Age')[['within_r_Fam1', f'within_r_{LABEL_B}']].describe().T)

# Between-infant: per age, pairwise r across infants' ROI mean waveforms
between_roi = {6: {'Fam1': [], LABEL_B: []}, 12: {'Fam1': [], LABEL_B: []}}
for age in (6, 12):
    infant_ids = [inf for (a, inf) in roi_hrfs if a == age]
    infant_ids.sort()
    for epoch in ('Fam1', LABEL_B):
        traces = [roi_hrfs[(age, inf)][epoch] for inf in infant_ids]
        for i in range(len(traces)):
            for j in range(i + 1, len(traces)):
                r = _pearson(traces[i][TIGHT], traces[j][TIGHT])
                if not np.isnan(r):
                    between_roi[age][epoch].append(r)

between_summary = []
for age in (6, 12):
    for epoch in ('Fam1', LABEL_B):
        vals = between_roi[age][epoch]
        between_summary.append({
            'Age': age, 'Epoch': epoch,
            'n_pairs': len(vals),
            'mean_r': float(np.mean(vals)) if vals else np.nan,
            'median_r': float(np.median(vals)) if vals else np.nan,
        })
betweenRoiSummaryDf = pd.DataFrame(between_summary)
report("\n=== Between-infant ROI pairwise r summary ===")
report(betweenRoiSummaryDf.to_string(index=False))


In [ ]:
# --- Plot within-infant and between-infant ROI variability ---

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex='col', sharey='row')

# Row 1: within-infant
bins = np.linspace(-1, 1, 41)
for col_idx, age in enumerate((6, 12)):
    ax = axes[0, col_idx]
    sub = withinRoiDf[withinRoiDf['Age'] == age]
    v1 = sub['within_r_Fam1'].dropna(); v3 = sub[f'within_r_{LABEL_B}'].dropna()
    ax.hist(v1, bins=bins, color='#00008B', alpha=0.6, edgecolor='white',
            label=f'Fam1 (n={len(v1)}, median={v1.median():.2f})')
    ax.hist(v3, bins=bins, color='#6495ED', alpha=0.6, edgecolor='white',
            label=f'{LABEL_B} (n={len(v3)}, median={v3.median():.2f})')
    ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
    ax.set_title(f'{age}mo: within-infant (trial-to-trial)', fontsize=10)
    ax.legend(fontsize=8, frameon=False)
axes[0, 0].set_ylabel('Count (infants)')

# Row 2: between-infant
for col_idx, age in enumerate((6, 12)):
    ax = axes[1, col_idx]
    for epoch, colour in [('Fam1', '#00008B'), (LABEL_B, '#6495ED')]:
        vals = between_roi[age][epoch]
        if not vals:
            continue
        ax.hist(vals, bins=bins, color=colour, alpha=0.55, edgecolor='white',
                label=f'{epoch}  (n={len(vals)} pairs, median={np.median(vals):.2f})')
    ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
    ax.set_title(f'{age}mo: between-infant (pairwise infant means)', fontsize=10)
    ax.set_xlabel('Pearson r')
    ax.legend(fontsize=8, frameon=False)
axes[1, 0].set_ylabel('Count (pairs)')

fig.suptitle(f"3.9. ROI-level within- and between-infant variability (threshold: '{ROI_THRESHOLD}')", y=1.00)
fig.tight_layout(); plt.show()

# Comparisons
report("\n--- Within-infant, Mann-Whitney (H1: 6mo > 12mo) ---")
for col in ('within_r_Fam1', f'within_r_{LABEL_B}'):
    v6  = withinRoiDf[withinRoiDf['Age'] == 6][col].dropna()
    v12 = withinRoiDf[withinRoiDf['Age'] == 12][col].dropna()
    u = mannwhitneyu(v6, v12, alternative='greater')
    report(f"{col}: U={u.statistic:.1f}, p={u.pvalue:.4g}, medians 6mo={v6.median():.3f}, 12mo={v12.median():.3f}")

report("\n--- Between-infant, Mann-Whitney (H1: 6mo > 12mo) ---")
for epoch in ('Fam1', LABEL_B):
    v6  = between_roi[6][epoch]
    v12 = between_roi[12][epoch]
    if v6 and v12:
        u = mannwhitneyu(v6, v12, alternative='greater')
        report(f"between_{epoch}: U={u.statistic:.1f}, p={u.pvalue:.4g}, medians 6mo={np.median(v6):.3f}, 12mo={np.median(v12):.3f}")


## 3.10 ROI overlap between infants (Jaccard similarity)

Do infants at each age share the same parcels? For each pair of infants at each
age, compute Jaccard similarity of their ROIs: |A ∩ B| / |A ∪ B|. Higher = more
spatial agreement.

**Prediction under individuation:** 6mo Jaccard higher than 12mo (6mo infants
converge on similar cortical territory; 12mo infants activate individually).

In [ ]:
# --- 3.10 ROI Jaccard similarity ---

def _jaccard(a, b):
    A = set(a); B = set(b)
    if not A and not B:
        return np.nan
    return len(A & B) / len(A | B) if (A | B) else np.nan


jaccard_rows = {6: [], 12: []}
for age in (6, 12):
    infants = sorted([inf for (a, inf) in personal_rois if a == age])
    for i in range(len(infants)):
        for j in range(i + 1, len(infants)):
            r = _jaccard(personal_rois[(age, infants[i])], personal_rois[(age, infants[j])])
            if not np.isnan(r):
                jaccard_rows[age].append(r)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
bins = np.linspace(0, 1, 41)
for age, colour in [(6, 'crimson'), (12, 'firebrick')]:
    vals = jaccard_rows[age]
    ax.hist(vals, bins=bins, color=colour, alpha=0.55, edgecolor='white',
            label=f'{age}mo  (n_pairs={len(vals)}, median={np.median(vals):.3f})')
ax.set_xlabel('Jaccard similarity between infants\' ROIs')
ax.set_ylabel('Count (pairs)')
ax.set_title(f"3.10. ROI Jaccard similarity between infants (threshold: '{ROI_THRESHOLD}')")
ax.legend(fontsize=9, frameon=False)
fig.tight_layout(); plt.show()

# Comparison
u = mannwhitneyu(jaccard_rows[6], jaccard_rows[12], alternative='greater')
report(f"Mann-Whitney U (H1: 6mo > 12mo Jaccard): U={u.statistic:.1f}, p={u.pvalue:.4g}")
report(f"  6mo:  n_pairs={len(jaccard_rows[6])}, median = {np.median(jaccard_rows[6]):.3f}")
report(f"  12mo: n_pairs={len(jaccard_rows[12])}, median = {np.median(jaccard_rows[12]):.3f}")


## 3.11 ROI centre-of-mass scatter

Where does each infant's ROI live anatomically? For each age:

1. Load the age-specific mesh and atlas.
2. Project each vertex through the atlas to get its parcel label.
3. Compute parcel centres-of-mass (mean XYZ of vertices assigned to that parcel).
4. For each infant, compute the centre-of-mass of their ROI (mean XYZ of the
   parcel centres in their ROI, unweighted).
5. Plot the 3D scatter of infants' ROI centres per age; report the median
   pairwise Euclidean distance between infants' ROI centres.

**Prediction under individuation:** 12mo infants' ROI centres are more scattered
than 6mo (larger median pairwise distance).

In [ ]:
# --- 3.11 Parcel coordinates extraction ---
# Mesh/atlas paths — mirror agePaths from brain_surface_plots_masked.py
AGE_PATHS = {
    6: {
        'atlas': '/Volumes/Extreme SSD/mri/registered/UNC_Surf_to_NeuroDev/No Mask/06mo/06mo_WangFunctional_Reg_Head.nii.gz',
        'mat':   '/Volumes/Extreme SSD/mri/meshes/corticalSurface06mo.mat',
    },
    12: {
        'atlas': '/Volumes/Extreme SSD/mri/registered/UNC_Surf_to_NeuroDev/No Mask/12mo/12mo_WangFunctional_Reg_Head.nii.gz',
        'mat':   '/Volumes/Extreme SSD/mri/meshes/corticalSurface12mo.mat',
    },
}


parcel_coords_rows = []
if _NIB_OK:
    for age, paths in AGE_PATHS.items():
        try:
            atlas_img = nib.load(paths['atlas'])
        except Exception as e:
            report(f"  {age}mo: failed to load atlas ({e}) - skipping")
            continue
        atlas_data = atlas_img.get_fdata()
        try:
            mat = loadmat(paths['mat'])
        except Exception as e:
            report(f"  {age}mo: failed to load mesh ({e}) - skipping")
            continue
        vertices = mat['vertices']
        inv_affine = np.linalg.inv(atlas_img.affine)
        voxel_coords = nib.affines.apply_affine(inv_affine, vertices)
        voxel_coords = np.round(voxel_coords).astype(int)
        voxel_coords = np.clip(voxel_coords, [0, 0, 0], np.array(atlas_data.shape) - 1)
        labels = np.array([atlas_data[x, y, z] for x, y, z in voxel_coords])
        # For vertices with label 0 (unlabelled), assign nearest labelled parcel
        unlabelled_idx = np.where(labels == 0)[0]
        labelled_idx = np.where(labels != 0)[0]
        if len(unlabelled_idx) > 0 and len(labelled_idx) > 0:
            tree = cKDTree(vertices[labelled_idx])
            _, nearest = tree.query(vertices[unlabelled_idx])
            labels[unlabelled_idx] = labels[labelled_idx][nearest]

        for parcel in np.unique(labels):
            if parcel == 0: continue
            mask = labels == parcel
            centre = vertices[mask].mean(axis=0)
            parcel_coords_rows.append({
                'Age': age, 'Parcel': int(parcel),
                'X': float(centre[0]), 'Y': float(centre[1]), 'Z': float(centre[2]),
            })
        report(f"  {age}mo: extracted centre-of-mass for {len(np.unique(labels))} parcels")

parcelCoordsDf = pd.DataFrame(parcel_coords_rows)
report(f"\nparcelCoordsDf: {len(parcelCoordsDf):,} (Age, Parcel) rows")
if not parcelCoordsDf.empty:
    parcelCoordsDf.to_csv(DATA + "parcel_centres_of_mass.csv", index=False)
    report(f"Saved parcel centres to {DATA}parcel_centres_of_mass.csv")


In [ ]:
# --- Compute each infant's ROI centre-of-mass and scatter ---

if not parcelCoordsDf.empty:
    coords_lookup = {(row['Age'], row['Parcel']): np.array([row['X'], row['Y'], row['Z']])
                     for _, row in parcelCoordsDf.iterrows()}

    roi_centres = {6: [], 12: []}
    for (age, infant), roi_parcels in personal_rois.items():
        pts = [coords_lookup[(age, p)] for p in roi_parcels if (age, p) in coords_lookup]
        if not pts:
            continue
        centre = np.mean(np.vstack(pts), axis=0)
        roi_centres[age].append({'Infant': infant, 'X': centre[0], 'Y': centre[1], 'Z': centre[2]})

    # Plot 3D scatter per age
    fig = plt.figure(figsize=(12, 5))
    for col_idx, age in enumerate((6, 12)):
        ax = fig.add_subplot(1, 2, col_idx + 1, projection='3d')
        pts = np.array([[r['X'], r['Y'], r['Z']] for r in roi_centres[age]])
        if len(pts) == 0:
            ax.set_title(f'{age}mo: no ROI centres')
            continue
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2],
                    c=('crimson' if age == 6 else 'firebrick'),
                    alpha=0.55, s=25)
        ax.set_title(f'{age}mo: ROI centres (n={len(pts)} infants)', fontsize=10)
        ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    fig.suptitle(f"3.11. ROI centre-of-mass scatter (threshold: '{ROI_THRESHOLD}')", y=1.02)
    fig.tight_layout(); plt.show()

    # Pairwise Euclidean distance between ROI centres per age
    for age in (6, 12):
        pts = np.array([[r['X'], r['Y'], r['Z']] for r in roi_centres[age]])
        if len(pts) < 2:
            continue
        dists = []
        for i in range(len(pts)):
            for j in range(i + 1, len(pts)):
                dists.append(np.linalg.norm(pts[i] - pts[j]))
        report(f"{age}mo: n_pairs={len(dists)}, "
              f"median pairwise ROI-centre distance = {np.median(dists):.2f}")

    # Compare between ages
    d6 = []
    pts6 = np.array([[r['X'], r['Y'], r['Z']] for r in roi_centres[6]])
    for i in range(len(pts6)):
        for j in range(i + 1, len(pts6)):
            d6.append(np.linalg.norm(pts6[i] - pts6[j]))
    d12 = []
    pts12 = np.array([[r['X'], r['Y'], r['Z']] for r in roi_centres[12]])
    for i in range(len(pts12)):
        for j in range(i + 1, len(pts12)):
            d12.append(np.linalg.norm(pts12[i] - pts12[j]))

    if d6 and d12:
        u = mannwhitneyu(d12, d6, alternative='greater')
        report(f"\nMann-Whitney U (H1: 12mo pairwise scatter > 6mo): U={u.statistic:.1f}, p={u.pvalue:.4g}")
else:
    report("parcelCoordsDf is empty - skipping ROI centre-of-mass analysis.")


## 3.12 Parcel-level infant-count distribution

Sections 3.4-3.11 have shown that individual-level habituation is present and equivalent at both ages, and that spatial individuation (as measured by ROI overlap and centre-of-mass scatter) is not the mechanism for the group-level direction-consistency/directional-hit-count failure at 12mo. This section tests one remaining possibility: whether 6mo infants converge on a small number of "hotspot" parcels (allowing the group test to fire at those hotspots) while 12mo infants distribute their activation more diffusely (so no single parcel accumulates enough infants to fire).

The direct measurement: for each (Age, Parcel), count how many infants activated that parcel. The direction-consistency/directional-hit-count test requires ≥6 infants per cell to run at all. If 6mo has more parcels reaching that threshold — or has parcels with higher infant counts — that could explain why 6mo group tests fire and 12mo don't.

**We test this for both activation (individual-level TFCE) and masked habituation, since the group-level direction-consistency/directional-hit-count test that failed was on habituation.**

In [ ]:
# --- 3.12a. Parcel-level infant-count distributions ---
# For each (Age, Parcel), count how many infants activated (or habituated at)
# that parcel. Compare distributions per age.

# --- Activation counts, from resultsDf using the current ROI_THRESHOLD ---
if ROI_THRESHOLD == 'fdr':
    act_sig = resultsDf[(resultsDf['Chrom'] == CHROM) & (resultsDf['Significant'] == True)]
elif ROI_THRESHOLD == 'uncorrected':
    act_sig = resultsDf[(resultsDf['Chrom'] == CHROM) & (resultsDf['p at peak (point)'] < 0.05)]

act_counts = (act_sig
              .groupby(['Age', 'Parcel'])['Infant']
              .nunique()
              .reset_index(name='n_infants'))

# --- Masked contrast counts ---
# From the masked contrast CSV produced by the individual-level pipeline
CONTRAST_TO_CSV_STUB = {
    'fam1_fam2': 'earlyHab',   
    'fam1_fam3': 'habituation',
    'delayed':   'delayedHab',
    'long_range': 'lrHab',
}
HAB_MASKED_CSV = DATA + f"surf_parcel_TFCE_{CONTRAST_TO_CSV_STUB[HAB_CONTRAST]}_MASKED_INDIV_spatiotemporal_perm_label.csv"
try:
    habMaskedDf = pd.read_csv(HAB_MASKED_CSV)
    hab_sig = habMaskedDf[(habMaskedDf['Chrom'] == CHROM)
                          & (habMaskedDf['Masked significant'] == True)]
    hab_counts = (hab_sig
                  .groupby(['Age', 'Parcel'])['Infant']
                  .nunique()
                  .reset_index(name='n_infants'))
except FileNotFoundError:
    report(f"WARNING: Masked hab CSV not found at {HAB_MASKED_CSV} — hab_counts will be empty")
    hab_counts = pd.DataFrame(columns=['Age', 'Parcel', 'n_infants'])

report("=== Activation: distribution of infant-count per (Age, Parcel) ===\n")
for age in (6, 12):
    sub = act_counts[act_counts['Age'] == age]
    n_parcels_active = len(sub)
    max_infants = sub['n_infants'].max() if not sub.empty else 0
    n_parcels_ge6 = int((sub['n_infants'] >= 6).sum())
    n_parcels_ge10 = int((sub['n_infants'] >= 10).sum())
    report(f"  {age}mo (activation, {CHROM_LABELS[CHROM]}, threshold={ROI_THRESHOLD}):")
    report(f"    parcels with any infant active:      {n_parcels_active}")
    report(f"    parcels with >= 6 infants active:    {n_parcels_ge6}")
    report(f"    parcels with >= 10 infants active:   {n_parcels_ge10}")
    report(f"    max infants at any single parcel:    {max_infants}")

report("\n=== Habituation (masked): distribution of infant-count per (Age, Parcel) ===\n")
for age in (6, 12):
    sub = hab_counts[hab_counts['Age'] == age]
    n_parcels_hab = len(sub)
    max_infants = sub['n_infants'].max() if not sub.empty else 0
    n_parcels_ge6 = int((sub['n_infants'] >= 6).sum())
    n_parcels_ge10 = int((sub['n_infants'] >= 10).sum())
    report(f"  {age}mo (masked {HAB_CONTRAST}, {CHROM_LABELS[CHROM]}):")
    report(f"    parcels with any infant habituating:      {n_parcels_hab}")
    report(f"    parcels with >= 6 infants habituating:    {n_parcels_ge6}")
    report(f"    parcels with >= 10 infants habituating:   {n_parcels_ge10}")
    report(f"    max infants at any single parcel:         {max_infants}")

In [ ]:
# --- 3.12b. Histograms of infant-count per parcel, per age ---

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey='row')

for col_idx, age in enumerate((6, 12)):
    # Activation
    ax = axes[0, col_idx]
    sub = act_counts[act_counts['Age'] == age]
    if not sub.empty:
        max_n = int(sub['n_infants'].max())
        ax.hist(sub['n_infants'], bins=np.arange(0.5, max_n + 1.5),
                color=('crimson' if age == 6 else 'firebrick'),
                alpha=0.7, edgecolor='white')
    ax.axvline(6, color='k', ls='--', lw=1, alpha=0.7, label='test direction-consistency/directional-hit-count n>=6 threshold')
    ax.set_title(f'{age}mo activation ({CHROM_LABELS[CHROM]}, thr="{ROI_THRESHOLD}")', fontsize=10)
    ax.set_xlabel('Number of infants activating parcel')
    ax.legend(fontsize=8, frameon=False)
    if col_idx == 0:
        ax.set_ylabel('Number of parcels')

    # Habituation
    ax = axes[1, col_idx]
    sub = hab_counts[hab_counts['Age'] == age]
    if not sub.empty:
        max_n = int(sub['n_infants'].max())
        ax.hist(sub['n_infants'], bins=np.arange(0.5, max_n + 1.5),
                color=('crimson' if age == 6 else 'firebrick'),
                alpha=0.7, edgecolor='white')
    ax.axvline(6, color='k', ls='--', lw=1, alpha=0.7, label='direction-consistency/directional-hit-count n>=6 threshold')
    ax.set_title(f'{age}mo masked {HAB_CONTRAST} ({CHROM_LABELS[CHROM]})', fontsize=10)
    ax.set_xlabel('Number of infants habituating at parcel')
    ax.legend(fontsize=8, frameon=False)
    if col_idx == 0:
        ax.set_ylabel('Number of parcels')

fig.suptitle('3.12b. Distribution of infant-counts per parcel', y=1.00)
fig.tight_layout(); plt.show()

In [ ]:
# --- 3.12c. Rank plot: parcels sorted by infant-count, per age ---
# Zipf-style: is 6mo's distribution more peaked (few hotspot parcels with many infants)
# than 12mo's? Or are they similar shapes?

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Activation
ax = axes[0]
for age, colour in [(6, 'crimson'), (12, 'firebrick')]:
    sub = act_counts[act_counts['Age'] == age].sort_values('n_infants', ascending=False)
    if sub.empty:
        continue
    ax.plot(np.arange(1, len(sub) + 1), sub['n_infants'].values,
            color=colour, lw=2,
            label=f'{age}mo  (n_parcels={len(sub)}, max={sub["n_infants"].max()})')
ax.axhline(6, color='k', ls='--', lw=1, alpha=0.7, label='n=6 (the directional-hit-count test threshold)')
ax.set_xlabel('Parcel rank (highest infant count first)')
ax.set_ylabel('Number of infants activating')
ax.set_title(f'Activation ranked by infant-count (thr="{ROI_THRESHOLD}")', fontsize=10)
ax.legend(fontsize=9, frameon=False)

# Habituation
ax = axes[1]
for age, colour in [(6, 'crimson'), (12, 'firebrick')]:
    sub = hab_counts[hab_counts['Age'] == age].sort_values('n_infants', ascending=False)
    if sub.empty:
        continue
    ax.plot(np.arange(1, len(sub) + 1), sub['n_infants'].values,
            color=colour, lw=2,
            label=f'{age}mo  (n_parcels={len(sub)}, max={sub["n_infants"].max()})')
ax.axhline(6, color='k', ls='--', lw=1, alpha=0.7, label='n=6 (the directional-hit-count test threshold)')
ax.set_xlabel('Parcel rank (highest infant count first)')
ax.set_ylabel('Number of infants habituating')
ax.set_title(f'Masked {HAB_CONTRAST} ranked by infant-count', fontsize=10)
ax.legend(fontsize=9, frameon=False)

fig.suptitle('3.12c. Rank of parcels by infant-count', y=1.02)
fig.tight_layout(); plt.show()

## 3.13 ROI-level direction-consistency/directional-hit-count analog

Use each infant's ROI-level habituation amplitude (Fam1 peak-window mean minus Fcontrast epoch peak-window mean, from Section 3.6) as their single per-infant observation. Then:

- **3.13a — sign-consistency binomial (direction-consistency analog).** Of infants with a valid hab_amp at each age, how many are negative (Fam1 < contrast in the hypothesised direction)? Test against 0.5 with a one-sided binomial. Analog of the direction-consistency test but at the ROI level.
- **3.13b — magnitude test (directional-hit-count analog).** Wilcoxon signed-rank test of hab_amp values against 0 (already done in 3.6, repeated here for completeness alongside 3.13a).
- **Between-age comparison.** Mann-Whitney U on hab_amp between ages.

Because each infant contributes one observation, no FDR correction is needed (only one test per age per analog). This is a stronger claim than the parcel-level direction-consistency/directional-hit-count because it uses each infant's aggregate response rather than per-parcel data.

In [ ]:
# --- 3.13 ROI-level direction-consistency/directional-hit-count analog ---
# habAmpDf from Section 3.6 gives us the per-infant Fam1-Contrast Epoch amplitude.
# We now run the group-level tests treating each infant as one observation.

results_3_13 = []
for age in (6, 12):
    vals = habAmpDf[habAmpDf['Age'] == age]['hab_amp'].dropna()
    if len(vals) == 0:
        continue
    n_total = len(vals)
    n_negative = int((vals < 0).sum())

    # 3.13a: sign-consistency binomial (one-sided, greater than 0.5)
    binom_direction_consistency = binomtest(n_negative, n_total, 0.5, alternative='greater')

    # 3.13b: Wilcoxon signed-rank against 0 (one-sided, greater than 0)
    w_directional_hit_count = wilcoxon(vals, alternative='greater')

    results_3_13.append({
        'Age': age,
        'n_infants': n_total,
        'n_negative': n_negative,
        'frac_negative': n_negative / n_total,
        'median_hab_amp': float(vals.median()),
        'p_direction_consistency_analog': float(binom_direction_consistency.pvalue),
        'p_directional_hit_count_analog': float(w_directional_hit_count.pvalue),
    })

results_3_13_df = pd.DataFrame(results_3_13)
report(f"=== 3.13 ROI-level direction-consistency/directional-hit-count analog (threshold: '{ROI_THRESHOLD}') ===\n")
report(results_3_13_df.to_string(index=False))
report()

# Between-age comparison
h6 = habAmpDf[habAmpDf['Age'] == 6]['hab_amp'].dropna()
h12 = habAmpDf[habAmpDf['Age'] == 12]['hab_amp'].dropna()
if len(h6) > 0 and len(h12) > 0:
    u_ages = mannwhitneyu(h6, h12, alternative='two-sided')
    report(f"Between-age comparison (Mann-Whitney U, two-sided): "
          f"U={u_ages.statistic:.1f}, p={u_ages.pvalue:.4g}")
    report(f"  6mo median = {h6.median():.4f}, 12mo median = {h12.median():.4f}")

## 3.14 Where does the habituation occur (spatial map)

Two complementary approaches:

**Approach A — Contribution-weighted spatial map (descriptive).** For each infant, compute their per-parcel Fam1-contrast epoch amplitude (only within their ROI parcels). Aggregate across infants at each age: how many infants had a negaitive Fam1-contrast amplitude at that parcel, weighted by amplitude. Produces a heatmap where cortical location = colour intensity of habituation.

**Approach B — ROI-restricted parcel-level test (inferential).** For each parcel, identify the infants who had that parcel in their personal ROI. If there are ≥ MIN_INFANTS_3_14 (see below) such infants, run a one-sided Wilcoxon test on their per-parcel Fam1-contrast epoch amplitudes. This is a proper inferential test that answers "at this parcel, does the population habituate — but only counting infants for whom this parcel is part of their ROI."

Approach B uses a lower `MIN_INFANTS_3_14` threshold than direction-consistency/directional-hit-count's ≥ 6 because we're operating on higher-SNR data (ROI-restricted). Suggested = 4-6; user can adjust.

In [ ]:
# --- 3.14 approach A: per-parcel habituation contribution across infants ---
# For each (Age, Infant, Parcel-in-their-ROI), compute Fam1-Contrast Epoch in the
# peak window. Then aggregate across infants at each parcel.

per_parcel_hab_rows = []
peak_slice = slice(PEAK_ONSET, PEAK_OFFSET)

for (age, infant), h in roi_hrfs.items():
    roi_parcels = h['roi_parcels']
    inf_meta = _meta_chrom[(_meta_chrom['Age'] == age) & (_meta_chrom['Infant'] == infant)
                          & (_meta_chrom['Parcel'].isin(roi_parcels))]
    for parcel in roi_parcels:
        parcel_meta = inf_meta[inf_meta['Parcel'] == parcel]
        fam1_traces = []
        fam3_traces = []
        for _, row in parcel_meta.iterrows():
            ts = tsDataset[row['tsIDx']] * 1e6
            if np.isnan(ts).all():
                continue
            if len(ts) > 231:
                ts = ts[:231]
            elif len(ts) < 231:
                ts = np.pad(ts, (0, 231 - len(ts)), constant_values=np.nan)
            if row['Trial'] in GROUP_A_TRIALS:
                fam1_traces.append(ts)
            elif row['Trial'] in GROUP_B_TRIALS:
                fam3_traces.append(ts)
        if not fam1_traces or not fam3_traces:
            continue
        fam1_mean = np.nanmean(fam1_traces, axis=0)
        fam3_mean = np.nanmean(fam3_traces, axis=0)
        fam1_pk = float(np.nanmean(fam1_mean[peak_slice]))
        fam3_pk = float(np.nanmean(fam3_mean[peak_slice]))
        per_parcel_hab_rows.append({
            'Age': age, 'Infant': infant, 'Parcel': parcel,
            'hab_amp': fam1_pk - fam3_pk,
        })

perParcelHabDf = pd.DataFrame(per_parcel_hab_rows)
report(f"Per-parcel habituation dataset: {len(perParcelHabDf):,} (Age, Infant, Parcel) rows\n")

# Aggregate to parcel level per age
parcel_agg_A = (perParcelHabDf
                 .groupby(['Age', 'Parcel'])
                 .agg(n_infants=('Infant', 'nunique'),
                      n_negative=('hab_amp', lambda x: int((x < 0).sum())),
                      median_hab_amp=('hab_amp', 'median'),
                      mean_hab_amp=('hab_amp', 'mean'))
                 .reset_index())
parcel_agg_A['frac_negative'] = parcel_agg_A['n_negative'] / parcel_agg_A['n_infants']

report("=== 3.14a. Per-parcel habituation summary (Approach A) ===")
report(f"6mo: {int((parcel_agg_A['Age']==6).sum())} unique parcels with any infant contribution")
report(f"12mo: {int((parcel_agg_A['Age']==12).sum())} unique parcels with any infant contribution")
report()

# Show top-10 hotspot parcels per age by mean hab_amp
for age in (6, 12):
    sub = parcel_agg_A[(parcel_agg_A['Age'] == age) & (parcel_agg_A['n_infants'] >= 3)]
    top = sub.sort_values('mean_hab_amp').head(10)
    report(f"--- {age}mo: top 10 parcels by mean habituation amplitude (n_infants >= 3) ---")
    report(top[['Parcel', 'n_infants', 'n_negative', 'mean_hab_amp', 'median_hab_amp']].to_string(index=False))
    report()

In [ ]:
# --- 3.14a plot: distribution of mean hab_amp per parcel, per age ---
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for age, colour in [(6, 'crimson'), (12, 'firebrick')]:
    sub = parcel_agg_A[(parcel_agg_A['Age'] == age) & (parcel_agg_A['n_infants'] >= 3)]
    vals = sub['mean_hab_amp']
    ax.hist(vals, bins=30, color=colour, alpha=0.55, edgecolor='white',
            label=f'{age}mo (n_parcels={len(sub)}, median={vals.median():.3f})')
ax.axvline(0, color='k', ls=':', lw=1, alpha=0.7)
ax.set_xlabel(f'Mean habituation amplitude across contributing infants ($\Delta$ {CHROM_LABELS[CHROM]} $\mu$M)')
ax.set_ylabel('Number of parcels')
ax.set_title(f"3.14a. Per-parcel mean habituation amplitude (n_infants >= 3, threshold: '{ROI_THRESHOLD}', {HAB_CONTRAST})")
ax.legend(fontsize=9, frameon=False)
fig.tight_layout(); plt.show()

In [ ]:
# --- 3.14 approach B: ROI-restricted parcel-level inferential test ---

MIN_INFANTS_3_14 = 4   # lower than direction-consistency/directional-hit-count's 6 because of higher-SNR ROI-restricted data

def _wilcoxon_greater(vals):
    if len(vals) < 3:
        return np.nan
    try:
        w = wilcoxon(vals, alternative='less')
        return float(w.pvalue)
    except ValueError:
        return np.nan

parcel_agg_B_rows = []
for (age, parcel), grp in perParcelHabDf.groupby(['Age', 'Parcel']):
    vals = grp['hab_amp'].dropna()
    if len(vals) < MIN_INFANTS_3_14:
        continue
    parcel_agg_B_rows.append({
        'Age': age, 'Parcel': parcel,
        'n_infants': len(vals),
        'median_hab_amp': float(vals.median()),
        'mean_hab_amp': float(vals.mean()),
        'p_wilcoxon': _wilcoxon_greater(vals.values),
    })

parcel_agg_B = pd.DataFrame(parcel_agg_B_rows)
report(f"=== 3.14b. Approach B — parcel-level Wilcoxon (n>={MIN_INFANTS_3_14}) ===\n")
for age in (6, 12):
    sub = parcel_agg_B[parcel_agg_B['Age'] == age].copy()
    if sub.empty:
        report(f"  {age}mo: no parcels reach n_infants >= {MIN_INFANTS_3_14}")
        continue
    # BH correction within age
    reject_bh, p_bh, _, _ = multipletests(sub['p_wilcoxon'].values, alpha=0.05, method='fdr_bh')
    sub['p_bh'] = p_bh
    sub['sig_bh'] = reject_bh
    n_tested = len(sub)
    n_sig_uncorr = int((sub['p_wilcoxon'] < 0.05).sum())
    n_sig_bh = int(reject_bh.sum())
    report(f"  {age}mo: {n_tested} parcels tested (n>={MIN_INFANTS_3_14})")
    report(f"    parcels significant uncorrected (p<0.05):  {n_sig_uncorr}")
    report(f"    parcels significant BH-corrected (q<0.05): {n_sig_bh}")
    if n_sig_bh > 0:
        top = sub[sub['sig_bh']].sort_values('p_bh').head(15)
        report("    Top BH-significant parcels:")
        report(top[['Parcel', 'n_infants', 'median_hab_amp', 'p_wilcoxon', 'p_bh']].to_string(index=False))
    report()

## 3.15 Group HRFs at BH-significant parcels

For each BH-significant parcel from 3.14b, plot the group-averaged Fam1 and contrast epochs HRFs across the infants who had that parcel in their ROI. Confirms that the parcel-level habituation amplitudes are driven by canonical HRF shapes rather than noise.

**Parameters:**
- `HRF_TOP_N`: number of BH-significant parcels to plot (sorted by significance). Set to `None` to plot ALL BH-significant parcels; then plots are paginated 20 per figure.
- Applied per age.

In [ ]:
# --- 3.15 Setup: which parcels to plot ---
HRF_TOP_N = None     # int, or None to plot ALL BH-significant parcels
HRF_PER_PAGE = 20  # subplots per figure page

# Extract BH-significant parcels per age from 3.14b's parcel_agg_B
sig_parcels_by_age = {}
for age in (6, 12):
    sub = parcel_agg_B[parcel_agg_B['Age'] == age].copy()
    if sub.empty:
        sig_parcels_by_age[age] = []
        continue
    
    reject_bh, p_bh, _, _ = multipletests(sub['p_wilcoxon'].values, alpha=0.05, method='fdr_bh')
    sub['p_bh'] = p_bh
    sub['sig_bh'] = reject_bh
    sig = sub[sub['sig_bh']].sort_values('p_bh')
    sig_parcels_by_age[age] = sig['Parcel'].tolist()

for age in (6, 12):
    report(f"{age}mo: {len(sig_parcels_by_age[age])} BH-significant parcels available for plotting")

# Compute group Fam1 and Contrast Epoch HRFs per BH-significant parcel, per age
# (across infants who had that parcel in their ROI)
def _parcel_group_hrf(age, parcel):
    """Return dict with fam1_mean, fam1_sem, fam3_mean, fam3_sem, n_infants."""
    per_inf_fam1 = []
    per_inf_fam3 = []
    for (a, infant), h in roi_hrfs.items():
        if a != age or parcel not in h['roi_parcels']:
            continue
        inf_meta = _meta_chrom[(_meta_chrom['Age'] == age) & (_meta_chrom['Infant'] == infant)
                              & (_meta_chrom['Parcel'] == parcel)]
        fam1_traces = []
        fam3_traces = []
        for _, row in inf_meta.iterrows():
            ts = tsDataset[row['tsIDx']] * 1e6
            if np.isnan(ts).all():
                continue
            if len(ts) > 231:
                ts = ts[:231]
            elif len(ts) < 231:
                ts = np.pad(ts, (0, 231 - len(ts)), constant_values=np.nan)
            if row['Trial'] in GROUP_A_TRIALS:
                fam1_traces.append(ts)
            elif row['Trial'] in GROUP_B_TRIALS:
                fam3_traces.append(ts)
        if fam1_traces and fam3_traces:
            per_inf_fam1.append(np.nanmean(fam1_traces, axis=0))
            per_inf_fam3.append(np.nanmean(fam3_traces, axis=0))
    if not per_inf_fam1:
        return None
    f1_stack = np.vstack(per_inf_fam1)
    f3_stack = np.vstack(per_inf_fam3)
    n = f1_stack.shape[0]
    return {
        'fam1_mean': np.nanmean(f1_stack, axis=0),
        'fam1_sem':  np.nanstd(f1_stack, axis=0) / np.sqrt(n),
        'fam3_mean': np.nanmean(f3_stack, axis=0),
        'fam3_sem':  np.nanstd(f3_stack, axis=0) / np.sqrt(n),
        'n_infants': n,
    }

In [ ]:
# --- 3.15 Plot group HRFs per BH-significant parcel, paginated ---
def _plot_parcel_hrfs_paginated(age, parcels_to_plot, per_page=HRF_PER_PAGE):
    if not parcels_to_plot:
        report(f"{age}mo: no parcels to plot.")
        return
    n_parcels = len(parcels_to_plot)
    n_pages = int(np.ceil(n_parcels / per_page))
    report(f"{age}mo: plotting {n_parcels} parcels across {n_pages} page(s)")

    for page in range(n_pages):
        page_parcels = parcels_to_plot[page*per_page : (page+1)*per_page]
        n_this_page = len(page_parcels)
        ncols = 4
        nrows = int(np.ceil(n_this_page / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(3.2*ncols, 2.2*nrows), sharey=True)
        axes = np.atleast_2d(axes).reshape(nrows, ncols)

        for idx, parcel in enumerate(page_parcels):
            row_idx = idx // ncols
            col_idx = idx % ncols
            ax = axes[row_idx, col_idx]
            hrf = _parcel_group_hrf(age, parcel)
            if hrf is None:
                ax.set_title(f'Parcel {parcel}\n(no data)', fontsize=8)
                continue
            for epoch, trace_key, sem_key in [('Fam1', 'fam1_mean', 'fam1_sem'),
                                                (LABEL_B, 'fam3_mean', 'fam3_sem')]:
                s = _style(CHROM, epoch)
                ax.plot(TIME_AXIS, hrf[trace_key], label=epoch, lw=1.3, **s)
                ax.fill_between(TIME_AXIS,
                                 hrf[trace_key] - hrf[sem_key],
                                 hrf[trace_key] + hrf[sem_key],
                                 color=s['color'], alpha=0.2, lw=0)
            ax.axhline(0, color='grey', lw=0.4, alpha=0.7)
            ax.axvline(0, color='grey', lw=0.4, ls=':', alpha=0.7)
            ax.axvspan((PEAK_ONSET - STIM_ONSET)/FS_HZ,
                       (PEAK_OFFSET - STIM_ONSET)/FS_HZ, color='yellow', alpha=0.08, lw=0)
            ax.set_title(f'Parcel {parcel}  (n={hrf["n_infants"]})', fontsize=8)
            ax.tick_params(labelsize=7)
            if row_idx == nrows - 1:
                ax.set_xlabel('Time (s)', fontsize=8)
            if col_idx == 0:
                ax.set_ylabel(f'$\Delta$ {CHROM_LABELS[CHROM]} ($\mu$M)', fontsize=8)
            if idx == 0 and page == 0:
                ax.legend(fontsize=7, frameon=False)

        # Hide unused subplots
        for empty_idx in range(n_this_page, nrows * ncols):
            axes.flat[empty_idx].axis('off')

        title_suffix = f' — page {page+1}/{n_pages}' if n_pages > 1 else ''
        fig.suptitle(
            f"3.15. {age}mo group-averaged HRFs at BH-significant parcels "
            f"(threshold: '{ROI_THRESHOLD}', {HAB_CONTRAST}){title_suffix}",
            y=1.00,
        )
        fig.tight_layout(); plt.show()

# --- Run ---
for age in (6, 12):
    parcels = sig_parcels_by_age[age]
    if HRF_TOP_N is not None:
        parcels = parcels[:HRF_TOP_N]
    _plot_parcel_hrfs_paginated(age, parcels)

## 3.16 Spatial map of BH-significant parcels

Two visualisations:

- **3.16A**: three separate mesh figures showing (i) 6mo BH-significant parcels only, (ii) 12mo BH-significant parcels only, (iii) parcels significant at both ages (intersection).
- **3.16B**: two combined-coded meshes (one per age) where each parcel is coloured by whether it is age-specific or shared with the other age.

Uses the same mesh/atlas infrastructure as `brain_surface_plots_masked.py`. Parcels are coloured by median habituation amplitude (from 3.14b). Non-significant parcels shown as translucent grey base.

In [ ]:
# --- 3.16 Setup: identify age-specific and shared BH-significant parcels ---
if PLOT_MESHES == 'yes':
      sig_6  = set(sig_parcels_by_age.get(6, []))
      sig_12 = set(sig_parcels_by_age.get(12, []))
      sig_both      = sig_6 & sig_12
      sig_6_only    = sig_6 - sig_12
      sig_12_only   = sig_12 - sig_6

      report(f"BH-significant parcels: 6mo={len(sig_6)}, 12mo={len(sig_12)}")
      report(f"  Shared:       {len(sig_both)} parcels: {sorted(sig_both)[:20]}"
            f"{'...' if len(sig_both) > 20 else ''}")
      report(f"  6mo only:     {len(sig_6_only)}")
      report(f"  12mo only:    {len(sig_12_only)}")

      # median hab_amp per (age, parcel) as the colour value
      hab_amp_lookup = {}
      for _, row in parcel_agg_B.iterrows():
            hab_amp_lookup[(int(row['Age']), int(row['Parcel']))] = float(row['median_hab_amp'])

In [ ]:
# --- 3.16 Mesh renderer, using the same approach as brain_surface_plots_masked.py ---

def _load_age_mesh(age, mesh_dir="/Volumes/Extreme SSD/mri/meshes/"):
    """Load vertices and faces for an age-specific cortical mesh."""
    path = f"{mesh_dir}corticalSurface{age:02d}mo.mat"
    try:
        mat = loadmat(path)
        vertices = mat['vertices']
        faces = mat['faces'] - 1  # MATLAB 1-based -> Python 0-based
        return vertices, faces
    except Exception as e:
        report(f"  Failed to load mesh for {age}mo: {e}")
        return None, None


def _get_vertex_labels(age, vertices):
    """Project vertices through the age-specific atlas to get parcel labels."""
    atlas_path = (f"/Volumes/Extreme SSD/mri/registered/UNC_Surf_to_NeuroDev/"
                  f"No Mask/{age:02d}mo/{age:02d}mo_WangFunctional_Reg_Head.nii.gz")
    try:
        atlas_img = nib.load(atlas_path)
    except Exception as e:
        report(f"  Failed to load atlas for {age}mo: {e}")
        return None
    atlas_data = atlas_img.get_fdata()
    inv_affine = np.linalg.inv(atlas_img.affine)
    voxel_coords = nib.affines.apply_affine(inv_affine, vertices)
    voxel_coords = np.round(voxel_coords).astype(int)
    voxel_coords = np.clip(voxel_coords, [0,0,0], np.array(atlas_data.shape) - 1)
    labels = np.array([atlas_data[x,y,z] for x,y,z in voxel_coords])
    # Assign nearest-labelled parcel to unlabelled vertices
    unlabelled = np.where(labels == 0)[0]
    labelled = np.where(labels != 0)[0]
    if len(unlabelled) > 0 and len(labelled) > 0:
        tree = cKDTree(vertices[labelled])
        _, nearest = tree.query(vertices[unlabelled])
        labels[unlabelled] = labels[labelled][nearest]
    return labels.astype(int)


def _plot_mesh_with_parcels(age, parcel_to_value_map, title,
                              default_colour_by='hab_amp'):
    """
    Render age's mesh; colour vertices in parcels listed in `parcel_to_value_map`
    by their value; non-highlighted vertices shown as translucent grey base.

    parcel_to_value_map: dict of parcel -> value (either numeric for cmap, or a
    category label for discrete colouring).
    """
    vertices, faces = _load_age_mesh(age)
    if vertices is None:
        return
    labels = _get_vertex_labels(age, vertices)
    if labels is None:
        return

    # Base grey mesh (all vertices)
    fig = go.Figure()
    fig.add_trace(go.Mesh3d(
        x=vertices[:,0], y=vertices[:,1], z=vertices[:,2],
        i=faces[:,0], j=faces[:,1], k=faces[:,2],
        color='lightgrey', opacity=0.3, showscale=False,
        lighting=dict(ambient=0.8),
    ))

    # Highlighted parcels
    if isinstance(next(iter(parcel_to_value_map.values()), None), str):
        # Categorical colouring
        if CHROM == 1:
            category_colours = {
                '6mo_only':  'crimson',
                '12mo_only': 'firebrick',
                'shared':    'gold',
                '6mo':       'crimson', '12mo': 'firebrick', 'intersection': 'gold',
            }
        else:
            category_colours = {
                '6mo_only':  'royalblue',
                '12mo_only': 'darkblue',
                'shared':    'gold',
                '6mo':       'royalblue', '12mo': 'darkblue', 'intersection': 'gold',
            }
        for category, colour in category_colours.items():
            parcels_in_cat = [p for p, v in parcel_to_value_map.items() if v == category]
            if not parcels_in_cat:
                continue
            mask = np.isin(labels, parcels_in_cat)
            sig_face_mask = mask[faces].all(axis=1)
            sig_faces = faces[sig_face_mask]
            if len(sig_faces) == 0:
                continue
            fig.add_trace(go.Mesh3d(
                x=vertices[:,0], y=vertices[:,1], z=vertices[:,2],
                i=sig_faces[:,0], j=sig_faces[:,1], k=sig_faces[:,2],
                color=colour, opacity=1.0, showscale=False,
                name=category, showlegend=True,
                lighting=dict(ambient=0.8),
            ))
    else:
        # Continuous colouring by hab_amp
        vertex_values = np.full(len(vertices), np.nan)
        for parcel, value in parcel_to_value_map.items():
            vertex_values[labels == parcel] = value
        valid_mask = ~np.isnan(vertex_values)
        sig_face_mask = valid_mask[faces].all(axis=1)
        sig_faces = faces[sig_face_mask]
        if len(sig_faces) > 0:
            values = vertex_values
            # Use gamma-weighted intensity, like brain_surface_plots_masked.py
            gamma = 0.65
            vals = np.where(np.isnan(values), 0, np.abs(values))
            intensity = vals ** gamma
            cmax_raw = float(max(abs(v) for v in parcel_to_value_map.values()))
            cmax_t = cmax_raw ** gamma
            fig.add_trace(go.Mesh3d(
                x=vertices[:,0], y=vertices[:,1], z=vertices[:,2],
                i=sig_faces[:,0], j=sig_faces[:,1], k=sig_faces[:,2],
                intensity=intensity, cmin=0, cmax=cmax_t,
                colorscale=(
                    [[0.0, 'rgb(255,220,220)'], [1.0, 'rgb(160,0,0)']] if CHROM == 1
                    else [[0.0, 'rgb(220,220,255)'], [1.0, 'rgb(0,0,160)']]
                ),
                opacity=1.0, showscale=True,
                colorbar=dict(title='Median hab_amp',
                              thickness=15, len=0.7, x=0.95),
                lighting=dict(ambient=0.8),
            ))

    # Compute mesh-size-appropriate figure dims
    extents = vertices.max(axis=0) - vertices.min(axis=0)
    max_extent = float(extents.max())
    width = int(np.clip(850 * max_extent / 130, 600, 1200))
    height = int(width * 0.85)
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False),
            bgcolor='white', aspectmode='data',
            camera=dict(eye=dict(x=0.9, y=0.9, z=0.9)),
        ),
        margin=dict(l=0, r=0, t=40, b=0),
        width=width, height=height,
    )
    fig.show()

In [ ]:
# --- 3.16A: three mesh figures (6mo, 12mo, intersection) ---
if PLOT_MESHES == 'yes':
    # 6mo BH-significant only, coloured by median hab_amp
    if sig_6:
        parcel_val_6 = {p: hab_amp_lookup.get((6, p), 0.0) for p in sig_6}
        _plot_mesh_with_parcels(6, parcel_val_6,
            f"3.16A. 6mo BH-significant parcels ({len(sig_6)} parcels, threshold: '{ROI_THRESHOLD}', {HAB_CONTRAST})")

    # 12mo BH-significant only, coloured by median hab_amp
    if sig_12:
        parcel_val_12 = {p: hab_amp_lookup.get((12, p), 0.0) for p in sig_12}
        _plot_mesh_with_parcels(12, parcel_val_12,
            f"3.16A. 12mo BH-significant parcels ({len(sig_12)} parcels, threshold: '{ROI_THRESHOLD}', {HAB_CONTRAST})")

    # Intersection: parcels significant at BOTH ages
    # Plot on the 12mo mesh (or the 6mo; doesn't matter much) — use average hab_amp
    if sig_both:
        parcel_val_both = {
            p: 0.5 * (hab_amp_lookup.get((6, p), 0.0) + hab_amp_lookup.get((12, p), 0.0))
            for p in sig_both
        }
        _plot_mesh_with_parcels(12, parcel_val_both,
            f"3.16A. Intersection: BH-significant at BOTH 6mo and 12mo "
            f"({len(sig_both)} parcels, plotted on 12mo mesh, {HAB_CONTRAST})")

In [ ]:
# --- 3.16B: two mesh figures with categorical colour-coding ---
if PLOT_MESHES == 'yes':
    # 6mo mesh: each 6mo-significant parcel is either "shared" or "6mo_only"
    parcel_category_6 = {}
    for p in sig_6:
        parcel_category_6[p] = 'shared' if p in sig_both else '6mo_only'
    if parcel_category_6:
        _plot_mesh_with_parcels(6, parcel_category_6,
            f"3.16B. 6mo BH-significant parcels colour-coded  "
            f"(6mo-only vs shared with 12mo, threshold: '{ROI_THRESHOLD}', {HAB_CONTRAST})")

    # 12mo mesh: each 12mo-significant parcel is either "shared" or "12mo_only"
    parcel_category_12 = {}
    for p in sig_12:
        parcel_category_12[p] = 'shared' if p in sig_both else '12mo_only'
    if parcel_category_12:
        _plot_mesh_with_parcels(12, parcel_category_12,
            f"3.16B. 12mo BH-significant parcels colour-coded  "
            f"(12mo-only vs shared with 6mo, threshold: '{ROI_THRESHOLD}', {HAB_CONTRAST})")

# Section 4 — mixed-effects group-level habituation

The section 3.13 test treats each infant as one observation (median hab_amp across their ROI). This section fits a mixed-effects model that uses each (infant, parcel) observation directly, with random intercepts for infant and parcel and SNR as a covariate. Uses Section 3 data (Section 3 must have been run for the current `ROI_THRESHOLD`).

**Model:** `hab_amp ~ Age + SNR + (1 | Infant) + (1 | Parcel)`

- Fixed effects: **Age** (categorical, 6mo vs 12mo) and **SNR** (per-observation Fam1 SNR at that parcel, centred).
- Random intercepts: **Infant** (habituation propensity varies across infants), **Parcel** (habituation strength varies across parcels).

**Predictions and interpretation:**
- Intercept (adjusted for Age and SNR): population habituation magnitude. > 0 (for HbO) or < 0 (for HbR) → group-level habituation is present.
- Age effect: developmental change in habituation. Not significant → habituation is equivalent across ages.
- SNR effect: does higher SNR predict larger hab_amp? Confirms measurement quality matters (or doesn't).

In [ ]:
# --- 4.1 Assemble the (Infant, Age, Parcel) dataset with hab_amp and SNR ---
# Per-parcel per-infant habituation amplitude (already exists as perParcelHabDf
# from section 3.14a). Per-infant per-parcel SNR needs to be computed.

# For each (Age, Infant, Parcel), get their Fam1 SNR at that specific parcel.
# SNR = peak / baseline SD at that parcel's per-infant Fam1 mean trace.
me_snr_rows = []
for (age, infant), h in roi_hrfs.items():
    inf_meta = _meta_chrom[(_meta_chrom['Age'] == age) & (_meta_chrom['Infant'] == infant)
                          & (_meta_chrom['Parcel'].isin(h['roi_parcels']))]
    for parcel in h['roi_parcels']:
        parcel_meta = inf_meta[inf_meta['Parcel'] == parcel]
        fam1_traces = []
        for _, row in parcel_meta.iterrows():
            if row['Trial'] not in GROUP_A_TRIALS:
                continue
            ts = tsDataset[row['tsIDx']] * 1e6
            if np.isnan(ts).all():
                continue
            if len(ts) > 231:
                ts = ts[:231]
            elif len(ts) < 231:
                ts = np.pad(ts, (0, 231 - len(ts)), constant_values=np.nan)
            fam1_traces.append(ts)
        if not fam1_traces:
            continue
        fam1_mean = np.nanmean(fam1_traces, axis=0)
        snr_val = _snr_of_mean(fam1_mean)   # from section 3.3
        me_snr_rows.append({
            'Age': age, 'Infant': infant, 'Parcel': parcel, 'snr_Fam1': snr_val,
        })

meSnrDf = pd.DataFrame(me_snr_rows)

# Merge with perParcelHabDf from section 3.14a
meDf = perParcelHabDf.merge(meSnrDf, on=['Age', 'Infant', 'Parcel'], how='inner')
meDf = meDf.dropna(subset=['hab_amp', 'snr_Fam1']).copy()

# Keep only 6mo and 12mo rows
meDf = meDf[meDf['Age'].isin([6, 12])].copy()

# Centre SNR
meDf['snr_Fam1_centred'] = meDf['snr_Fam1'] - meDf['snr_Fam1'].mean()

# Age as string (not Categorical — causes statsmodels issues)
meDf['Age_cat'] = meDf['Age'].map({6: '6mo', 12: '12mo'})
meDf['Age_cat'] = pd.Categorical(meDf['Age_cat'], categories=['6mo', '12mo'], ordered=False)

# Sign-flip for HbR
HAB_SIGN = +1 if CHROM == 1 else -1
meDf['hab_amp_canonical'] = HAB_SIGN * meDf['hab_amp']

# Clean sequential index
meDf = meDf.reset_index(drop=True)

report(f"Mixed-effects dataset: {len(meDf):,} (Infant, Age, Parcel) rows")
report(f"  Unique infants: {meDf['Infant'].nunique()}")
report(f"  Unique parcels: {meDf['Parcel'].nunique()}")
report(f"  Rows per age: {meDf.groupby('Age_cat', observed=True).size().to_dict()}")
report(f"  Mean centred SNR: {meDf['snr_Fam1_centred'].mean():.3f} (should be ~0)")

In [ ]:
# --- 4.2 Fit mixed-effects model ---
# Two crossed random intercepts (Infant, Parcel) using variance-components syntax
vc_formula = {"Parcel": "0 + C(Parcel)"}

# Fit: hab_amp ~ Age + SNR + (1|Infant) + (1|Parcel)
mixedlm_model = smf.mixedlm(
    formula="hab_amp_canonical ~ Age_cat + snr_Fam1_centred",
    data=meDf,
    groups="Infant",                 # random intercept for Infant
    vc_formula=vc_formula,           # random intercept for Parcel via variance components
    re_formula="1",                  # random intercept only (no random slope)
)

# Fit with REML (default) — better for random-effect variance estimation
mixedlm_result = mixedlm_model.fit(reml=True, method='lbfgs')

report("=" * 70)
report("Mixed-effects model: hab_amp ~ Age + SNR + (1|Infant) + (1|Parcel)")
report("=" * 70)
report(mixedlm_result.summary())

In [ ]:
# --- 4.3 Extract fixed-effect estimates + interpretation ---
fe = mixedlm_result.fe_params
se = mixedlm_result.bse_fe
pvals = mixedlm_result.pvalues
cis = mixedlm_result.conf_int()

report("\n=== Fixed effects summary ===")
for name in fe.index:
    coef = fe[name]
    stderr = se[name]
    p = pvals[name]
    ci_lo = cis.loc[name, 0]
    ci_hi = cis.loc[name, 1]
    report(f"  {name:35s} = {coef:+.4f}  (SE {stderr:.4f}, 95% CI [{ci_lo:+.4f}, {ci_hi:+.4f}], p = {p:.4g})")

# Random-effect variances
report(f"\n=== Random-effect variances ===")
report(f"  Infant variance:   {mixedlm_result.cov_re.iloc[0, 0]:.6f}")
if hasattr(mixedlm_result, 'vcomp') and len(mixedlm_result.vcomp) > 0:
    report(f"  Parcel variance:   {mixedlm_result.vcomp[0]:.6f}")
report(f"  Residual variance: {mixedlm_result.scale:.6f}")

# Interpretive summary
intercept = fe.get('Intercept', np.nan)

# Detect which age is the reference category based on the coefficient name
age_coef_names = [n for n in fe.index if 'Age_cat' in n]
if age_coef_names:
    age_coef_name = age_coef_names[0]  # e.g. 'Age_cat[T.12mo]'
    contrast_age = age_coef_name.split('T.')[1].rstrip(']')
    reference_age = '12mo' if contrast_age == '6mo' else '6mo'
    age_effect = fe[age_coef_name]
    age_pval = pvals[age_coef_name]
else:
    contrast_age = reference_age = None
    age_effect = np.nan
    age_pval = np.nan

snr_effect = fe.get('snr_Fam1_centred', np.nan)

report(f"\n=== Interpretation (contrast: {HAB_CONTRAST}, chromophore: {CHROM_LABELS[CHROM]}) ===")
report(f"  Population habituation (at {reference_age or 'reference age'}, mean SNR): {intercept:+.4f} μM")
if contrast_age:
    report(f"  Additional habituation at {contrast_age} (vs {reference_age}): "
           f"{age_effect:+.4f} μM (p = {age_pval:.4g})")
report(f"  Habituation per unit SNR increase: {snr_effect:+.4f} μM "
       f"(p = {pvals.get('snr_Fam1_centred', np.nan):.4g})")

In [ ]:
# Filter every array to just fixed-effect rows using fe.index
fe_names = list(fe.index)

fe_summary = pd.DataFrame({
    'chromophore':    [CHROM_LABELS[CHROM]] * len(fe_names),
    'contrast':       [HAB_CONTRAST]        * len(fe_names),
    'roi_threshold':  [ROI_THRESHOLD]       * len(fe_names),
    'coef_name':      fe_names,
    'coef':           [fe[n]    for n in fe_names],
    'se':             [se[n]    for n in fe_names],
    'p_value':        [pvals[n] for n in fe_names],
    'ci_lo':          [cis.loc[n, 0] for n in fe_names],
    'ci_hi':          [cis.loc[n, 1] for n in fe_names],
})
out_path = DATA + f"mixedeffects_summary_{CHROM_LABELS[CHROM]}_{HAB_CONTRAST}_{ROI_THRESHOLD}.csv"
fe_summary.to_csv(out_path, index=False)
print(f"Written: {out_path}")

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests

# ---------------------------------------------------------------------
# Load 6 mixed-effects result files and stack
# ---------------------------------------------------------------------
DATA = "/Users/sambe/Documents/Python_Projects/FDA/data/"

# Fixed configuration for this comparison
ROI_THRESHOLD = 'fdr'

contrasts = ['fam1_fam3', 'delayed', 'long_range']
chroms = ['HbO', 'HbR']

all_rows = []
for chrom in chroms:
    for contrast in contrasts:
        path = DATA + f"mixedeffects_summary_{chrom}_{contrast}_{ROI_THRESHOLD}.csv"
        try:
            df = pd.read_csv(path)
            all_rows.append(df)
        except FileNotFoundError:
            print(f"MISSING: {path}")
            continue

if not all_rows:
    raise FileNotFoundError(
        "No mixed-effects summary files found. Run cell 81 for each combination first."
    )

allDf = pd.concat(all_rows, ignore_index=True)

# This cell is cross-chromophore: it needs BOTH postHocMechanisticTestsHbO.ipynb
# and postHocMechanisticTestsHbR.ipynb to have been run first (so both
# chromophores' mixedeffects_summary_*.csv files exist on disk) - it will run
# from EITHER notebook, but only produces a complete result if both have been
# run. Explicitly check both chromophores are actually present, rather than
# just "at least one file was found":
present_chroms = set(allDf['chromophore'].unique()) if 'chromophore' in allDf.columns else set()
missing_chroms = set(chroms) - present_chroms
if missing_chroms:
    print(f"WARNING: missing results for chromophore(s) {sorted(missing_chroms)}. "
          f"Run the corresponding postHocMechanisticTests notebook(s) first, then "
          f"re-run this cell for the complete cross-chromophore comparison.")


# ---------------------------------------------------------------------
# Apply BH + Bonferroni correction within each fixed effect family
# (Intercept, Age, SNR): 6 tests per family, corrected separately.
# ---------------------------------------------------------------------
def _tag(name):
    """Return short tag for the fixed effect based on the coefficient name."""
    if name == 'Intercept':
        return 'Intercept'
    if 'Age_cat' in name:
        return 'Age'
    if 'snr' in name.lower():
        return 'SNR'
    return None


allDf['effect_family'] = allDf['coef_name'].apply(_tag)
corrected = allDf[allDf['effect_family'].notna()].copy()

# Correct within each family independently
corrected['p_bh']         = np.nan
corrected['p_bonf']       = np.nan
corrected['sig_bh_005']   = False
corrected['sig_bonf_005'] = False

for family, group_idx in corrected.groupby('effect_family').groups.items():
    pvals = corrected.loc[group_idx, 'p_value'].values
    reject_bh,   p_bh,   _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')
    reject_bonf, p_bonf, _, _ = multipletests(pvals, alpha=0.05, method='bonferroni')
    corrected.loc[group_idx, 'p_bh']         = p_bh
    corrected.loc[group_idx, 'p_bonf']       = p_bonf
    corrected.loc[group_idx, 'sig_bh_005']   = reject_bh
    corrected.loc[group_idx, 'sig_bonf_005'] = reject_bonf


# ---------------------------------------------------------------------
# Report per family
# ---------------------------------------------------------------------
cols_to_show = ['chromophore', 'contrast', 'coef', 'p_value',
                'p_bh', 'p_bonf', 'sig_bh_005', 'sig_bonf_005']

for family in ('Intercept', 'Age', 'SNR'):
    sub = corrected[corrected['effect_family'] == family].copy()
    n_tests = len(sub)
    print("=" * 100)
    print(f"{family} effect across chromophore x contrast (n_tests = {n_tests})")
    print("=" * 100)
    print(sub[cols_to_show].to_string(index=False))
    print()
    print(f"  Nominally significant (p < 0.05):       "
          f"{(sub['p_value'] < 0.05).sum()}/{n_tests}")
    print(f"  Significant after BH (q < 0.05):        "
          f"{sub['sig_bh_005'].sum()}/{n_tests}")
    print(f"  Significant after Bonferroni (< 0.05):  "
          f"{sub['sig_bonf_005'].sum()}/{n_tests}")
    print()


# ---------------------------------------------------------------------
# Optional: save the corrected results for the paper
# ---------------------------------------------------------------------
out_path = DATA + f"mixedeffects_corrected_across_chrom_contrast_{ROI_THRESHOLD}.csv"
corrected.to_csv(out_path, index=False)
print(f"Written: {out_path}")